# K Pro v1 — Pitcher Strikeout Prop Model
**System:** Pitcher Strikeout Props  
**Model:** XGBoost Poisson + Monte Carlo  
**Target:** Projected strikeout count vs DraftKings O/U + Ladder  
**Status:** Paper trading (gate: 200 settled bets)  

---
**Section Map**
| # | Section | Run when |
|---|---|---|
| 0 | Config & Imports | Always first |
| 1 | Statcast Load | Daily nightly |
| 2 | Umpire Load | Initial + nightly |
| 3 | DK Odds Scraper | Daily pre-game |
| 4 | Pitcher Feature Aggregation | After Statcast pull |
| 5 | Lineup Feature Join | After Section 4 |
| 6 | Feature Join Layer | After 4+5 |
| 7 | Model Train (OOS + Walk-forward CV) | Periodic — MANUAL GATE |
| 7b | Full Retrain (production) | Pre-season — MANUAL GATE |
| 8 | Monte Carlo K Distribution Engine | After training |
| 9 | Mathematical Rigor Assessment | After retrain |
| 10 | Backtest Engine | After training |
| 11 | Bet Tracker (SQLite) | Log + settle |
| 12a | Quick Refresh | Any time |
| 12b | Daily Maintenance | Once each morning |

## Section 0 — Config & Imports

In [1]:
import os, re, json, csv, time, math, sqlite3, unicodedata, warnings, atexit
import numpy as np
import pandas as pd
import xgboost as xgb
import requests
from datetime import date, timedelta, datetime
from pathlib import Path
from scipy.stats import kstest, nbinom

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE            = Path(r"C:\Users\lmayn\Downloads")
K_DIR           = BASE / "mlb-betting" / "K_Pro_System"
DATA_DIR        = K_DIR / "data"
MODEL_DIR       = K_DIR / "models"
STATCAST_MASTER = BASE / "Baseball_Data" / "Statcast" / "statcast_master.csv"
UMPIRE_DIR      = BASE / "Baseball_Data" / "Umpires"
DK_ODDS_DIR     = BASE / "Baseball_Data" / "Odds" / "DraftKings"

for d in [DATA_DIR, MODEL_DIR, DK_ODDS_DIR, UMPIRE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── XGBoost params ────────────────────────────────────────────────────────────
XGB_PARAMS = {
    "objective":        "count:poisson",
    "eval_metric":      "poisson-nloglik",
    "max_depth":        4,
    "learning_rate":    0.03,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 15,
    "reg_alpha":        1.0,
    "reg_lambda":       3.0,
    "gamma":            0.5,
    "seed":             42,
}

# ── Feature list (34 features) ────────────────────────────────────────────────
K_FEATURES = [
    # K rate
    "k_pct_L5", "k_pct_L10", "k_pct_STD", "k_per_9_L5", "k_per_9_L10",
    # Count leverage
    "first_pitch_strike_pct_L10", "hitter_count_rate_L10", "two_strike_k_rate_L10",
    # Stuff
    "whiff_pct_L10", "zone_contact_pct_L10", "chase_pct_L10", "velo_mean_L5", "velo_trend_L5",
    # Pitch mix
    "fb_pct_L10", "breaking_pct_L10", "primary_whiff_rate_L10",
    # Durability
    "avg_ip_L5", "avg_bf_L5", "days_rest", "short_rest",
    # Opponent
    "opp_k_rate_L14", "opp_k_rate_vs_hand_L14", "opp_chase_rate_L14", "opp_whiff_rate_L14",
    "opp_lineup_pct_L", "opp_platoon_k_edge", "opp_top3_k_rate_L50",
    # Umpire
    "ump_overall_accuracy_L30", "ump_k_boost_L30", "ump_consistency_L30",
    # Context
    "is_home", "implied_win_pct", "temperature_f", "is_dome",
]

# ── cfg dict ──────────────────────────────────────────────────────────────────
cfg = {
    "statcast_master":  str(STATCAST_MASTER),
    "pitcher_features": str(DATA_DIR / "pitcher_k_features.csv"),
    "lineup_features":  str(DATA_DIR / "lineup_k_features.csv"),
    "model_features":   str(DATA_DIR / "model_features.csv"),
    "k_odds_master":    str(DK_ODDS_DIR / "k_odds_master.csv"),
    "umpire_dir":       str(UMPIRE_DIR),
    "db_path":          str(K_DIR / "k_bets.db"),
    "model_oos":        str(MODEL_DIR / "xgb_k_v1_oos.json"),
    "model_prod":       str(MODEL_DIR / "xgb_k_v1.json"),
    "model_meta":       str(MODEL_DIR / "xgb_k_v1_meta.json"),
    "bankroll":         1000.0,
    "kelly_fraction":   0.25,
    "min_edge":         0.04,
    "min_kelly_pct":    0.005,
    "max_kelly_pct":    0.05,
    "max_ladder_rungs": 3,
    "paper_mode":       True,   # NEVER remove — gate until 200 settled bets
    "mc_sims":          10_000,
    "mc_cap":           14,     # Hard cap: physical MLB ceiling
    "cv_folds":         [2023, 2024, 2025],
    "train_start":      2021,
}

model = {}   # runtime model store: {"bst": xgb.Booster, "features": [...]}
data  = {}   # runtime data store

# ── Dome venues ───────────────────────────────────────────────────────────────
DOME_TEAMS = {"MIA", "HOU", "TB", "SEA", "TOR", "MIL", "ARI", "LAD", "MIN",
              "ATL", "TEX", "WSH"}

# ── Stadium coordinates for weather ──────────────────────────────────────────
STADIUM_COORDS = {
    "NYY": (40.8296, -73.9262), "NYM": (40.7571, -73.8458),
    "BOS": (42.3467, -71.0972), "BAL": (39.2838, -76.6216),
    "TB":  (27.7683, -82.6534), "TOR": (43.6414, -79.3894),
    "CLE": (41.4962, -81.6852), "DET": (42.3390, -83.0485),
    "CWS": (41.8300, -87.6338), "MIN": (44.9817, -93.2778),
    "KC":  (39.0517, -94.4803), "HOU": (29.7572, -95.3555),
    "TEX": (32.7512, -97.0832), "OAK": (37.7516, -122.2005),
    "SEA": (47.5914, -122.3325), "LAA": (33.8003, -117.8827),
    "ATL": (33.8908, -84.4678), "MIA": (25.7781, -80.2197),
    "WSH": (38.8730, -77.0074), "PHI": (39.9061, -75.1665),
    "NYM": (40.7571, -73.8458), "PIT": (40.4469, -80.0058),
    "CHC": (41.9484, -87.6553), "CIN": (39.0979, -84.5082),
    "MIL": (43.0280, -87.9712), "STL": (38.6226, -90.1928),
    "LAD": (34.0739, -118.2400), "SF":  (37.7786, -122.3893),
    "SD":  (32.7076, -117.1570), "COL": (39.7559, -104.9942),
    "ARI": (33.4453, -112.0667),
}

# ── DK team name maps ─────────────────────────────────────────────────────────
DK_NAME_TO_ABBR = {
    "New York Yankees": "NYY", "New York Mets": "NYM",
    "Boston Red Sox": "BOS", "Baltimore Orioles": "BAL",
    "Tampa Bay Rays": "TB", "Toronto Blue Jays": "TOR",
    "Cleveland Guardians": "CLE", "Detroit Tigers": "DET",
    "Chicago White Sox": "CWS", "Minnesota Twins": "MIN",
    "Kansas City Royals": "KC", "Houston Astros": "HOU",
    "Texas Rangers": "TEX", "Oakland Athletics": "OAK",
    "Seattle Mariners": "SEA", "Los Angeles Angels": "LAA",
    "Atlanta Braves": "ATL", "Miami Marlins": "MIA",
    "Washington Nationals": "WSH", "Philadelphia Phillies": "PHI",
    "Pittsburgh Pirates": "PIT", "Chicago Cubs": "CHC",
    "Cincinnati Reds": "CIN", "Milwaukee Brewers": "MIL",
    "St. Louis Cardinals": "STL", "Los Angeles Dodgers": "LAD",
    "San Francisco Giants": "SF", "San Diego Padres": "SD",
    "Colorado Rockies": "COL", "Arizona Diamondbacks": "ARI",
    # Abbreviation variants DK sometimes uses
    "NYY": "NYY", "NYM": "NYM", "BOS": "BOS", "BAL": "BAL",
    "TB": "TB", "TOR": "TOR", "CLE": "CLE", "DET": "DET",
    "CWS": "CWS", "MIN": "MIN", "KC": "KC", "HOU": "HOU",
    "TEX": "TEX", "OAK": "OAK", "SEA": "SEA", "LAA": "LAA",
    "ATL": "ATL", "MIA": "MIA", "WSH": "WSH", "PHI": "PHI",
    "PIT": "PIT", "CHC": "CHC", "CIN": "CIN", "MIL": "MIL",
    "STL": "STL", "LAD": "LAD", "SF": "SF", "SD": "SD",
    "COL": "COL", "ARI": "ARI",
}

# ── Shared utility functions ───────────────────────────────────────────────────

def american_to_implied_prob(odds) -> float:
    """Convert American odds (int) to implied probability."""
    odds = int(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return abs(odds) / (abs(odds) + 100.0)

def implied_to_american(prob: float) -> int:
    """Convert probability to American odds."""
    prob = max(0.001, min(0.999, prob))
    if prob >= 0.5:
        return int(round(-(prob / (1 - prob)) * 100))
    else:
        return int(round(((1 - prob) / prob) * 100))

def remove_vig(p_a: float, p_b: float):
    """Remove vig from two-sided market. Returns fair probs (sum to 1.0)."""
    total = p_a + p_b
    if total <= 0:
        return 0.5, 0.5
    return p_a / total, p_b / total

def kelly_pct(edge: float, odds: int, fraction: float = 0.25) -> float:
    """Full Kelly percentage (scaled by fraction). Returns 0 if edge <= 0."""
    if edge <= 0:
        return 0.0
    odds = int(odds)
    if odds > 0:
        b = odds / 100.0
    else:
        b = 100.0 / abs(odds)
    p = edge + american_to_implied_prob(odds)  # crude: market_p + edge
    q = 1.0 - p
    raw = (b * p - q) / b
    return max(0.0, raw * fraction)

def kelly_stake(edge: float, odds: int, bankroll: float, fraction: float,
                min_pct: float, max_pct: float) -> float:
    """Kelly stake in dollars, clamped to [min_pct, max_pct] of bankroll."""
    pct = kelly_pct(edge, odds, fraction)
    if pct < min_pct:
        return 0.0
    pct = min(pct, max_pct)
    return round(bankroll * pct, 2)

def normalize_name(s: str) -> str:
    """NFD unicode normalize, lowercase, strip punctuation/whitespace."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z ]", "", s.lower()).strip()
    return re.sub(r"\s+", " ", s)

def build_date_list(start: str, end: str = None):
    """Returns list of date strings YYYY-MM-DD from start to end (inclusive)."""
    if end is None:
        end = str(date.today())
    d0 = datetime.strptime(start, "%Y-%m-%d").date()
    d1 = datetime.strptime(end, "%Y-%m-%d").date()
    out = []
    while d0 <= d1:
        out.append(str(d0))
        d0 += timedelta(days=1)
    return out

def _dk_to_int(s) -> int:
    """Convert DK odds string to int. Handles Unicode minus characters."""
    if s is None:
        return None
    s = str(s).replace("\u2212", "-").replace("\u2013", "-").replace("\u2014", "-").strip()
    try:
        return int(s)
    except (ValueError, TypeError):
        return None

def _dk_resolve(name: str) -> str:
    """DK team name → standard abbreviation."""
    if not name:
        return name
    return DK_NAME_TO_ABBR.get(name, name)

print("Section 0 loaded. Paths initialized.")
print(f"  K_DIR:           {K_DIR}")
print(f"  STATCAST_MASTER: {STATCAST_MASTER}")
print(f"  paper_mode:      {cfg['paper_mode']}")

Section 0 loaded. Paths initialized.
  K_DIR:           C:\Users\lmayn\Downloads\mlb-betting\K_Pro_System
  STATCAST_MASTER: C:\Users\lmayn\Downloads\Baseball_Data\Statcast\statcast_master.csv
  paper_mode:      True


## Section 1 — Statcast Load

In [2]:
def load_statcast_k(cfg: dict) -> pd.DataFrame:
    """
    Load statcast_master.csv (all innings) for K Pro.
    Stores in data['statcast']. Returns DataFrame.
    Field availability:
      balls, strikes    — 2021+  (count-leverage features)
      outs_when_up      — 2021+  (IP calculation)
      bat_speed         — 2024+ ONLY — excluded from features entirely
    """
    master_path = cfg["statcast_master"]
    if not Path(master_path).exists():
        print(f"[WARN] Statcast master not found: {master_path}")
        data["statcast"] = pd.DataFrame()
        return data["statcast"]

    print(f"Loading Statcast master from {master_path} ...")
    df = pd.read_csv(master_path, low_memory=False)

    # Numeric coercion for fields that may be partially missing
    numeric_cols = [
        "release_speed", "pfx_x", "pfx_z", "balls", "strikes",
        "outs_when_up", "spin_axis", "release_extension", "effective_speed",
        "release_pos_x", "release_pos_z",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # bat_speed: hard exclude — 2024+ only, breaks walk-forward CV
    if "bat_speed" in df.columns:
        df = df.drop(columns=["bat_speed"])

    # Parse game_date
    if "game_date" in df.columns:
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

    # Filter to train_start (2021+)
    if "game_date" in df.columns:
        df = df[df["game_date"].dt.year >= cfg["train_start"]].copy()

    data["statcast"] = df

    n_rows    = len(df)
    date_min  = df["game_date"].min() if "game_date" in df.columns else "?"
    date_max  = df["game_date"].max() if "game_date" in df.columns else "?"
    n_pitchers = df["pitcher"].nunique() if "pitcher" in df.columns else "?"
    n_games   = df["game_pk"].nunique() if "game_pk" in df.columns else "?"

    print(f"  Rows:     {n_rows:,}")
    print(f"  Dates:    {date_min} → {date_max}")
    print(f"  Pitchers: {n_pitchers:,}")
    print(f"  Games:    {n_games:,}")
    return df


# ── Auto-run ───────────────────────────────────────────────────────────────────
load_statcast_k(cfg)

Loading Statcast master from C:\Users\lmayn\Downloads\Baseball_Data\Statcast\statcast_master.csv ...
  Rows:     921,985
  Dates:    2021-04-01 00:00:00 → 2026-04-01 00:00:00
  Pitchers: 1,756
  Games:    12,236


,pitcher,batter,player_name,game_pk,game_date,home_team,away_team,p_throws,stand,inning,...,release_pos_z,pfx_x,pfx_z,plate_x,plate_z,sz_top,sz_bot,zone,type,swing_length
0,594835,643289,"Gonzales, Marco",634625,2021-04-01,SEA,SF,L,R,6,...,5.92,1.10,1.76,0.111264,3.971739,3.371000,1.535000,12.0,X,NaN
1,676051,405395,"Heuer, Codi",634655,2021-04-01,LAA,CWS,R,R,7,...,5.87,-1.18,1.20,-0.471994,3.149858,3.490000,1.601000,1.0,X,NaN
2,572193,546990,"Tepera, Ryan",634634,2021-04-01,CHC,PIT,R,R,8,...,5.68,-0.55,1.40,0.722703,3.191807,3.411000,1.565000,3.0,S,NaN
3,622075,605131,"Almonte, Yency",634615,2021-04-01,COL,LAD,R,R,7,...,5.70,-0.66,0.63,-0.220931,2.080136,3.301000,1.504000,5.0,X,NaN
4,595465,607732,"Winkler, Dan",634634,2021-04-01,CHC,PIT,R,R,7,...,5.02,1.00,0.47,0.805922,2.227724,3.575000,1.681000,9.0,S,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
921980,676979,514888,"Crochet, Garrett",824211,2026-04-01,HOU,BOS,L,R,2,...,6.06,1.39,0.35,0.427182,2.481640,2.926001,1.476673,6.0,X,7.4
921981,657746,677951,"Ryan, Joe",824136,2026-04-01,KC,MIN,R,R,2,...,5.02,1.44,-0.57,0.373061,1.495425,3.238435,1.634350,14.0,X,7.5
921982,645261,695657,"Alcantara, Sandy",823889,2026-04-01,MIA,CWS,R,L,4,...,5.97,-1.48,0.15,-0.426956,1.809514,3.340240,1.685728,7.0,X,7.8
921983,592332,687859,"Gausman, Kevin",822835,2026-04-01,TOR,COL,R,L,4,...,5.83,-1.19,0.10,-0.665719,0.841863,3.159449,1.594488,13.0,S,8.2


## Section 2 — Umpire Load

In [7]:
def load_umpires(cfg: dict) -> pd.DataFrame:
    """
    Load UmpScorecards JSON files from umpire_dir.
    Computes rolling umpire metrics (L30) with shift(1).
    ump_k_boost_L30 is derived from Statcast (not UmpScorecards — no K data there).
    Stores in data['umps']. Returns DataFrame.
    """
    ump_dir = Path(cfg["umpire_dir"])
    json_files = sorted(ump_dir.glob("*.json"))
    if not json_files:
        print(f"[WARN] No UmpScorecards JSON files found in {ump_dir}")
        data["umps"] = pd.DataFrame()
        return data["umps"]

    rows = []
    for jf in json_files:
        try:
            with open(jf, "r", encoding="utf-8") as f:
                obj = json.load(f)
            file_rows = obj.get("rows", obj) if isinstance(obj, dict) else obj
            rows.extend(file_rows)
        except Exception as e:
            print(f"[WARN] Failed to load {jf.name}: {e}")

    if not rows:
        print("[WARN] UmpScorecards: no rows parsed.")
        data["umps"] = pd.DataFrame()
        return data["umps"]

    df = pd.DataFrame(rows)

    # Filter to regular season, parse date
    if "type" in df.columns:
        df = df[df["type"] == "R"].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

    # Ensure numeric
    for col in ["overall_accuracy", "consistency", "total_run_impact",
                "home_pitcher_impact", "away_pitcher_impact"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── Derive ump_k_boost_L30 from Statcast ───────────────────────────────────
    ump_k_boost = None
    if "statcast" in data and len(data["statcast"]) > 0:
        sc = data["statcast"]
        if "events" in sc.columns and "game_pk" in sc.columns:
            # PA-level rows only (events not null)
            pa_df = sc[sc["events"].notna()].copy()
            league_k_rate = (pa_df["events"] == "strikeout").mean()

            game_k = (
                pa_df.groupby("game_pk")
                .apply(lambda g: pd.Series({
                    "actual_ks":  (g["events"] == "strikeout").sum(),
                    "total_bf":   len(g)
                }))
                .reset_index()
            )
            game_k["expected_ks"] = game_k["total_bf"] * league_k_rate
            game_k["k_delta"]     = game_k["actual_ks"] - game_k["expected_ks"]

            if "game_pk" in df.columns:
                df["game_pk"] = pd.to_numeric(df["game_pk"], errors="coerce")
                df = df.merge(game_k[["game_pk", "k_delta"]], on="game_pk", how="left")
            else:
                df["k_delta"] = np.nan

            print(f"  League K rate: {league_k_rate:.4f}")
        else:
            df["k_delta"] = np.nan
    else:
        df["k_delta"] = np.nan

    # ── Rolling L30 metrics per umpire (shift(1), min_periods=5) ──────────────
    df = df.sort_values(["umpire", "date"]).reset_index(drop=True)

    def _roll30(series):
        return series.shift(1).rolling(30, min_periods=5).mean()

    df["ump_overall_accuracy_L30"] = df.groupby("umpire")["overall_accuracy"].transform(_roll30)
    df["ump_consistency_L30"]      = df.groupby("umpire")["consistency"].transform(_roll30)
    df["ump_k_boost_L30"]          = df.groupby("umpire")["k_delta"].transform(_roll30)

    # Re-sort by date for joins
    df = df.sort_values("date").reset_index(drop=True)

    data["umps"] = df
    print(f"  Umpire records: {len(df):,}  |  Umpires: {df['umpire'].nunique() if 'umpire' in df.columns else '?'}")
    print(f"  Date range: {df['date'].min()} → {df['date'].max()}")
    return df


# ── Auto-run ───────────────────────────────────────────────────────────────────
load_umpires(cfg)

  League K rate: 0.2257
  Umpire records: 7,170  |  Umpires: 103
  Date range: 2023-03-30 00:00:00 → 2025-09-28 00:00:00


,game_pk,failed,has_basic_game_data,has_detailed_game_data,fully_valid,num_pitches_no_data,below_missing_cutoff,asterisk,ND,date,...,favor,home_batter_impact,home_pitcher_impact,away_batter_impact,away_pitcher_impact,total_run_impact,k_delta,ump_overall_accuracy_L30,ump_consistency_L30,ump_k_boost_L30
0,718768,False,True,True,True,0,True,False,False,2023-03-30,...,0.20,-0.08,0.28,-0.28,0.08,0.36,5.296488,NaN,NaN,NaN
1,718777,False,True,True,True,0,True,False,False,2023-03-30,...,0.10,-0.15,0.25,-0.25,0.15,1.28,1.425103,NaN,NaN,NaN
2,718776,False,True,True,True,0,True,False,False,2023-03-30,...,-0.61,-0.03,-0.58,0.58,0.03,1.29,-2.123450,NaN,NaN,NaN
3,718781,False,True,True,True,0,True,False,False,2023-03-30,...,-1.24,-0.36,-0.88,0.88,0.36,2.30,16.876550,NaN,NaN,NaN
4,718779,False,True,True,True,0,True,False,False,2023-03-30,...,-0.77,-1.30,0.53,-0.53,1.30,1.99,3.393595,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7165,776140,False,True,True,True,0,True,False,False,2025-09-28,...,0.77,0.82,-0.05,0.05,-0.82,1.63,2.876550,93.501409,93.579689,-0.745248
7166,776136,False,True,True,True,0,True,False,False,2025-09-28,...,-0.44,-0.12,-0.32,0.32,0.12,0.88,0.490702,94.842857,94.244070,0.600775
7167,776144,False,True,True,True,0,True,False,False,2025-09-28,...,0.33,0.40,-0.07,0.07,-0.40,1.25,5.522211,94.585847,93.883305,-0.827049
7168,776137,False,True,True,True,0,True,False,False,2025-09-28,...,-0.52,-0.19,-0.33,0.33,0.19,1.24,3.102273,93.627315,93.387803,-0.645334


## Section 3 — DraftKings Odds Scraper

**Two separate Selenium sessions** — O/U and Ladder subcategory URLs are different pages.
Never combine into one session.

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import base64

DK_OU_URL     = ("https://sportsbook.draftkings.com/leagues/baseball/mlb"
                 "?category=pitcher-props&subcategory=strikeouts-o-u")
DK_LADDER_URL = ("https://sportsbook.draftkings.com/leagues/baseball/mlb"
                 "?category=pitcher-props&subcategory=strikeouts")


def _build_driver(headless: bool = True):
    """Build headless Chrome driver with CDP network logging enabled."""
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--log-level=3")
    opts.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    driver = webdriver.Chrome(options=opts)
    driver.execute_cdp_cmd("Network.enable", {})
    return driver


def _dk_fetch_payloads(url: str, wait: int = 8) -> list:
    """
    Selenium headless Chrome + CDP Network.getResponseBody.
    Returns list of parsed JSON payload dicts from DK sportscontent API.
    """
    driver = _build_driver(headless=True)
    payloads = []
    try:
        driver.get(url)
        time.sleep(wait)

        logs = driver.get_log("performance")
        request_ids = []
        for entry in logs:
            try:
                msg = json.loads(entry["message"])["message"]
                if msg.get("method") == "Network.responseReceived":
                    req_url = msg["params"]["response"]["url"]
                    if any(k in req_url for k in ["sportscontent", "sportsbook", "feature"]):
                        request_ids.append(msg["params"]["requestId"])
            except Exception:
                pass

        for req_id in request_ids:
            try:
                resp = driver.execute_cdp_cmd(
                    "Network.getResponseBody", {"requestId": req_id}
                )
                body = resp.get("body", "")
                if resp.get("base64Encoded"):
                    body = base64.b64decode(body).decode("utf-8", errors="ignore")
                parsed = json.loads(body)
                payloads.append(parsed)
            except Exception:
                pass
    finally:
        driver.quit()

    return payloads


def _dk_parse_combined(payloads: list):
    """
    Parse DK payload list into (events, markets, selections) dicts.
    Returns three flat dicts keyed by id.
    """
    events     = {}
    markets    = {}
    selections = {}

    for payload in payloads:
        if not isinstance(payload, dict):
            continue
        # Walk nested structures to find eventGroup / events / markets / selections
        def _walk(obj):
            if isinstance(obj, dict):
                if "eventGroupId" in obj or "homeTeam" in obj:
                    eid = obj.get("eventId") or obj.get("id")
                    if eid:
                        events[str(eid)] = obj
                if "marketType" in obj and "marketId" in obj:
                    markets[str(obj["marketId"])] = obj
                if "outcomeType" in obj or "milestoneValue" in obj:
                    sel_id = obj.get("selectionId") or obj.get("id")
                    if sel_id:
                        selections[str(sel_id)] = obj
                for v in obj.values():
                    _walk(v)
            elif isinstance(obj, list):
                for item in obj:
                    _walk(item)
        _walk(payload)

    return events, markets, selections


def _extract_player_and_team(selection: dict, events: dict) -> tuple:
    """Extract pitcher name and team from a DK selection."""
    pitcher_name = None
    team = None
    opponent = None

    # Pitcher name from participants
    parts = selection.get("participants", [])
    for p in parts:
        if p.get("type") == "Player" or "name" in p:
            pitcher_name = p.get("name")
            venue_role = p.get("venueRole", "")

            # Resolve team from event participants
            event_id = str(selection.get("eventId", ""))
            ev = events.get(event_id, {})
            ev_parts = ev.get("teamParticipants", ev.get("participants", []))
            for ep in ev_parts:
                role = ep.get("venueRole", "")
                raw_name = ep.get("name", ep.get("teamName", ""))
                abbr = _dk_resolve(raw_name) or raw_name
                if venue_role == "HomePlayer" and "Home" in role:
                    team = abbr
                elif venue_role == "AwayPlayer" and "Away" in role:
                    team = abbr
            # Opponent
            for ep in ev_parts:
                raw_name = ep.get("name", ep.get("teamName", ""))
                abbr = _dk_resolve(raw_name) or raw_name
                if abbr != team:
                    opponent = abbr
            break

    return pitcher_name, team, opponent


def _scrape_dk_ou() -> dict:
    """
    Scrape DK O/U strikeout lines.
    Returns {normalized_pitcher_name: {pitcher_name, team, opponent, k_line, over_odds, under_odds}}
    """
    print("Scraping DK O/U strikeout lines...")
    payloads = _dk_fetch_payloads(DK_OU_URL, wait=8)
    events, markets, selections = _dk_parse_combined(payloads)

    results = {}
    for sel_id, sel in selections.items():
        market_id = str(sel.get("marketId", ""))
        mkt = markets.get(market_id, {})
        mkt_type = mkt.get("marketType", {})
        mkt_name = mkt_type.get("name", "") if isinstance(mkt_type, dict) else str(mkt_type)

        if "Strikeouts Thrown O/U" not in mkt_name and "Strikeouts O/U" not in mkt_name:
            continue

        outcome = sel.get("outcomeType", "")
        if outcome not in ("Over", "Under"):
            continue

        k_line = sel.get("points")
        if k_line is None:
            continue
        try:
            k_line = float(k_line)
        except (ValueError, TypeError):
            continue

        odds_raw = (sel.get("displayOdds") or {}).get("american", "")
        odds = _dk_to_int(odds_raw)
        if odds is None:
            continue

        pitcher_name, team, opponent = _extract_player_and_team(sel, events)
        if not pitcher_name:
            continue

        key = normalize_name(pitcher_name)
        if key not in results:
            results[key] = {
                "pitcher_name": pitcher_name, "team": team, "opponent": opponent,
                "k_line": k_line, "over_odds": None, "under_odds": None
            }

        if outcome == "Over":
            results[key]["over_odds"] = odds
            results[key]["k_line"]    = k_line
        else:
            results[key]["under_odds"] = odds

    print(f"  O/U lines found: {len(results)}")
    return results


def _scrape_dk_ladder() -> dict:
    """
    Scrape DK strikeout milestone ladder.
    Returns {normalized_pitcher_name: {pitcher_name, team, opponent, k_3plus: odds, k_4plus: odds, ...}}
    Stores ALL rungs found (k_1plus through k_14plus). DK typically starts at k_3plus.
    """
    print("Scraping DK ladder strikeout lines...")
    payloads = _dk_fetch_payloads(DK_LADDER_URL, wait=8)
    events, markets, selections = _dk_parse_combined(payloads)

    results = {}
    for sel_id, sel in selections.items():
        market_id = str(sel.get("marketId", ""))
        mkt = markets.get(market_id, {})
        mkt_type = mkt.get("marketType", {})
        mkt_name = mkt_type.get("name", "") if isinstance(mkt_type, dict) else str(mkt_type)

        if "Milestone" not in mkt_name and "milestone" not in mkt_name.lower():
            continue

        # milestoneValue is on the selection, not the market
        milestone = sel.get("milestoneValue")
        if milestone is None:
            continue
        try:
            milestone = int(milestone)
        except (ValueError, TypeError):
            continue

        if not (1 <= milestone <= 14):
            continue

        odds_raw = (sel.get("displayOdds") or {}).get("american", "")
        odds = _dk_to_int(odds_raw)
        if odds is None:
            continue

        pitcher_name, team, opponent = _extract_player_and_team(sel, events)
        if not pitcher_name:
            continue

        key = normalize_name(pitcher_name)
        if key not in results:
            results[key] = {"pitcher_name": pitcher_name, "team": team, "opponent": opponent}

        results[key][f"k_{milestone}plus"] = odds

    print(f"  Ladder markets found: {len(results)}")
    return results


def scrape_dk_k_props(cfg: dict, target_date: str = None) -> pd.DataFrame:
    """
    Scrape DK O/U and ladder K props (two separate sessions).
    Merges on normalize_name(pitcher_name).
    Appends to k_odds_master.csv. Stores in data['k_odds'].
    Returns DataFrame.
    """
    if target_date is None:
        target_date = str(date.today())

    ou_dict     = _scrape_dk_ou()
    ladder_dict = _scrape_dk_ladder()

    # Merge on normalized name
    all_keys = set(ou_dict.keys()) | set(ladder_dict.keys())
    rows = []
    for key in all_keys:
        ou  = ou_dict.get(key, {})
        ldr = ladder_dict.get(key, {})

        row = {"date": target_date}
        row["pitcher_name"] = ou.get("pitcher_name") or ldr.get("pitcher_name", "")
        row["team"]         = ou.get("team") or ldr.get("team", "")
        row["opponent"]     = ou.get("opponent") or ldr.get("opponent", "")
        row["k_line"]       = ou.get("k_line")
        row["over_odds"]    = ou.get("over_odds")
        row["under_odds"]   = ou.get("under_odds")
        # Ladder rungs — store all k_1plus through k_14plus
        for n in range(1, 15):
            row[f"k_{n}plus"] = ldr.get(f"k_{n}plus")
        rows.append(row)

    df = pd.DataFrame(rows)

    # Append to master (no dedup — multiple pulls kept for CLV tracking)
    master_path = cfg["k_odds_master"]
    if Path(master_path).exists():
        df.to_csv(master_path, mode="a", header=False, index=False)
    else:
        df.to_csv(master_path, index=False)

    data["k_odds"] = df
    print(f"  Scraped {len(df)} pitchers for {target_date}. Appended to {master_path}.")
    return df


# ── Manual run ─────────────────────────────────────────────────────────────────
# odds_df = scrape_dk_k_props(cfg)
print("Section 3 loaded. Call scrape_dk_k_props(cfg) to scrape.")

Section 3 loaded. Call scrape_dk_k_props(cfg) to scrape.


## Section 4 — Pitcher Feature Aggregation

In [5]:
def _identify_starters(sc: pd.DataFrame) -> pd.DataFrame:
    """
    Identify starting pitcher per game: pitcher with most batters faced (PA-ending events).
    Returns DataFrame with columns: [game_pk, game_date, pitcher, starter_ks, starter_ip,
                                      starter_bf, p_throws, home_team, away_team, is_home]
    """
    # PA-ending rows
    pa = sc[sc["events"].notna()].copy()

    # Batters faced per pitcher per game
    bf = pa.groupby(["game_pk", "pitcher"]).size().reset_index(name="bf")

    # Starter = max bf per game
    idx     = bf.groupby("game_pk")["bf"].idxmax()
    starters = bf.loc[idx].copy()

    # Merge game_date
    gdates = sc.groupby("game_pk")["game_date"].first().reset_index()
    starters = starters.merge(gdates, on="game_pk", how="left")

    # Starter Ks
    starter_ks = (
        pa[pa["events"] == "strikeout"]
        .groupby(["game_pk", "pitcher"])
        .size()
        .reset_index(name="starter_ks")
    )
    starters = starters.merge(starter_ks, on=["game_pk", "pitcher"], how="left")
    starters["starter_ks"] = starters["starter_ks"].fillna(0).astype(int)

    # Starter IP from outs_when_up
    if "outs_when_up" in sc.columns:
        ip_df = (
            sc.groupby(["game_pk", "pitcher"])["outs_when_up"]
            .max()
            .reset_index()
        )
        ip_df["starter_ip"] = (ip_df["outs_when_up"] + 1) / 3.0
        starters = starters.merge(ip_df[["game_pk", "pitcher", "starter_ip"]],
                                   on=["game_pk", "pitcher"], how="left")
    else:
        starters["starter_ip"] = 5.0  # fallback

    starters["starter_ip"] = starters["starter_ip"].fillna(5.0)

    # p_throws (pitcher handedness)
    if "p_throws" in sc.columns:
        pt = sc.groupby("pitcher")["p_throws"].first().reset_index()
        starters = starters.merge(pt, on="pitcher", how="left")
    else:
        starters["p_throws"] = "R"

    # home/away from inning_topbot
    if "inning_topbot" in sc.columns and "home_team" in sc.columns:
        home = sc.groupby("game_pk")[["home_team", "away_team"]].first().reset_index() \
               if "away_team" in sc.columns else \
               sc.groupby("game_pk")["home_team"].first().reset_index()
        starters = starters.merge(home, on="game_pk", how="left")
    elif "home_team" in sc.columns:
        home = sc.groupby("game_pk")["home_team"].first().reset_index()
        starters = starters.merge(home, on="game_pk", how="left")

    return starters


def _build_pitch_game_aggs(sc: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate pitch-level Statcast to per-pitcher-per-game metrics.
    Returns DataFrame with one row per (game_pk, pitcher).
    """
    df = sc.copy()

    # Swing/miss flags
    df["is_swing"]          = df["description"].isin(["swinging_strike", "swinging_strike_blocked",
                                                        "foul", "foul_tip", "hit_into_play"]).astype(int) \
                              if "description" in df.columns else 0
    df["is_whiff"]          = df["description"].isin(["swinging_strike", "swinging_strike_blocked"]).astype(int) \
                              if "description" in df.columns else 0
    df["is_in_zone"]        = (df["zone"].between(1, 9)).astype(int) if "zone" in df.columns else 0
    df["is_out_zone"]       = (df["zone"] > 9).astype(int) if "zone" in df.columns else 0
    df["is_zone_contact"]   = (df["is_in_zone"] & ~df["is_whiff"].astype(bool) & df["is_swing"].astype(bool)).astype(int)
    df["is_chase_whiff"]    = (df["is_out_zone"] & df["is_whiff"].astype(bool)).astype(int)

    # First pitch strike (balls==0, strikes==0)
    if "balls" in df.columns and "strikes" in df.columns:
        df["is_first_pitch"] = ((df["balls"] == 0) & (df["strikes"] == 0)).astype(int)
        df["is_fps"]         = (df["is_first_pitch"] & df["is_in_zone"]).astype(int)
        # Hitter counts: 2-0, 3-0, 3-1
        df["is_hitter_count"] = (((df["balls"] == 2) & (df["strikes"] == 0)) |
                                  ((df["balls"] == 3) & (df["strikes"] == 0)) |
                                  ((df["balls"] == 3) & (df["strikes"] == 1))).astype(int)
        # Two-strike plate appearances
        df["is_two_strike"] = (df["strikes"] == 2).astype(int)
    else:
        df["is_first_pitch"] = 0
        df["is_fps"]         = 0
        df["is_hitter_count"] = 0
        df["is_two_strike"]   = 0

    # Pitch type flags
    if "pitch_type" in df.columns:
        df["is_fb"]      = df["pitch_type"].isin(["FF", "SI"]).astype(int)
        df["is_breaking"] = df["pitch_type"].isin(["SL", "CU", "KC", "CS"]).astype(int)
    else:
        df["is_fb"] = 0
        df["is_breaking"] = 0

    # Per-game, per-pitcher aggregation
    def _agg_game(g):
        n        = len(g)
        n_swing  = g["is_swing"].sum()
        n_whiff  = g["is_whiff"].sum()
        n_in_zone   = g["is_in_zone"].sum()
        n_zone_con  = g["is_zone_contact"].sum()
        n_out_zone  = g["is_out_zone"].sum()
        n_chase_wh  = g["is_chase_whiff"].sum()
        n_fp        = g["is_first_pitch"].sum()
        n_fps       = g["is_fps"].sum()
        n_hc        = g["is_hitter_count"].sum()
        n_2s        = g["is_two_strike"].sum()

        # two_strike_k_rate — fraction of two-strike PA that end in strikeout
        pa_2s = g[g["is_two_strike"] == 1]
        pa_2s_terminal = pa_2s[pa_2s["events"].notna()] if "events" in pa_2s.columns else pa_2s.head(0)
        two_k = (pa_2s_terminal["events"] == "strikeout").sum() / max(len(pa_2s_terminal), 1)

        # Primary pitch whiff rate
        pri_whiff = np.nan
        if "pitch_type" in g.columns and n > 0:
            pt_counts = g["pitch_type"].value_counts()
            if len(pt_counts) > 0:
                primary_pt = pt_counts.index[0]
                mask = g["pitch_type"] == primary_pt
                pri_n_swing = g.loc[mask, "is_swing"].sum()
                pri_n_whiff = g.loc[mask, "is_whiff"].sum()
                pri_whiff   = pri_n_whiff / max(pri_n_swing, 1)

        return pd.Series({
            "n_pitches":           n,
            "whiff_pct_game":      n_whiff / max(n_swing, 1),
            "zone_contact_pct":    n_zone_con / max(n_in_zone, 1),
            "chase_pct_game":      n_chase_wh / max(n_out_zone, 1),
            "fps_pct":             n_fps / max(n_fp, 1),
            "hitter_count_pct":    n_hc / max(n, 1),
            "two_strike_k_rate":   two_k,
            "fb_pct":              g["is_fb"].sum() / max(n, 1),
            "breaking_pct":        g["is_breaking"].sum() / max(n, 1),
            "velo_mean":           g["release_speed"].mean() if "release_speed" in g.columns else np.nan,
            "primary_whiff_rate":  pri_whiff,
        })

    pitch_agg = df.groupby(["game_pk", "pitcher"]).apply(_agg_game).reset_index()
    return pitch_agg


def _rolling_pitcher(df: pd.DataFrame, col: str, window: int,
                      min_p: int = 3, expand: bool = False) -> pd.Series:
    """
    Rolling mean of `col` per pitcher, shift(1) to prevent leakage.
    expand=True → expanding season-to-date mean.
    """
    def _r(s):
        s = s.shift(1)
        if expand:
            return s.expanding(min_periods=min_p).mean()
        return s.rolling(window, min_periods=min_p).mean()
    return df.groupby("pitcher")[col].transform(_r)


def build_pitcher_k_features(cfg: dict) -> pd.DataFrame:
    """
    Build one row per starter per game with all rolling K features.
    Stores in data['pitcher_features'], saves to pitcher_k_features.csv.
    """
    if "statcast" not in data or len(data["statcast"]) == 0:
        print("[WARN] Statcast not loaded. Run Section 1 first.")
        data["pitcher_features"] = pd.DataFrame()
        return data["pitcher_features"]

    sc = data["statcast"].copy()
    print("Building pitcher K features...")

    # 1. Identify starters
    starters = _identify_starters(sc)
    starters = starters.sort_values(["pitcher", "game_date"]).reset_index(drop=True)

    # 2. Pitch-level game aggregates
    pitch_agg = _build_pitch_game_aggs(sc)
    starters  = starters.merge(pitch_agg, on=["game_pk", "pitcher"], how="left")

    # 3. K rate inputs
    starters["k_pct"]   = starters["starter_ks"] / starters["bf"].replace(0, np.nan)
    starters["k_per_9"] = starters["starter_ks"] / starters["starter_ip"].replace(0, np.nan) * 9

    # 4. Rolling features — ALL shift(1) enforced by _rolling_pitcher
    starters = starters.sort_values(["pitcher", "game_date"]).reset_index(drop=True)

    # K rate
    starters["k_pct_L5"]   = _rolling_pitcher(starters, "k_pct",   5)
    starters["k_pct_L10"]  = _rolling_pitcher(starters, "k_pct",  10)
    starters["k_pct_STD"]  = _rolling_pitcher(starters, "k_pct",  10, expand=True)
    starters["k_per_9_L5"] = _rolling_pitcher(starters, "k_per_9", 5)
    starters["k_per_9_L10"]= _rolling_pitcher(starters, "k_per_9",10)

    # Count leverage
    starters["first_pitch_strike_pct_L10"] = _rolling_pitcher(starters, "fps_pct",         10)
    starters["hitter_count_rate_L10"]      = _rolling_pitcher(starters, "hitter_count_pct", 10)
    starters["two_strike_k_rate_L10"]      = _rolling_pitcher(starters, "two_strike_k_rate",10)

    # Stuff
    starters["whiff_pct_L10"]        = _rolling_pitcher(starters, "whiff_pct_game",    10)
    starters["zone_contact_pct_L10"] = _rolling_pitcher(starters, "zone_contact_pct",  10)
    starters["chase_pct_L10"]        = _rolling_pitcher(starters, "chase_pct_game",    10)
    starters["velo_mean_L5"]         = _rolling_pitcher(starters, "velo_mean",          5)

    # Velocity trend (slope over L5)
    def _velo_trend(s):
        def _slope(vals):
            v = vals.dropna()
            if len(v) < 3:
                return np.nan
            return np.polyfit(range(len(v)), v, 1)[0]
        return s.shift(1).rolling(5, min_periods=3).apply(_slope, raw=False)

    starters["velo_trend_L5"] = starters.groupby("pitcher")["velo_mean"].transform(_velo_trend)

    # Pitch mix
    starters["fb_pct_L10"]            = _rolling_pitcher(starters, "fb_pct",           10)
    starters["breaking_pct_L10"]      = _rolling_pitcher(starters, "breaking_pct",     10)
    starters["primary_whiff_rate_L10"]= _rolling_pitcher(starters, "primary_whiff_rate",10)

    # Durability
    starters["avg_ip_L5"] = _rolling_pitcher(starters, "starter_ip", 5)
    starters["avg_bf_L5"] = _rolling_pitcher(starters, "bf",         5)

    starters["days_rest"] = (
        starters.groupby("pitcher")["game_date"]
        .transform(lambda s: s.diff().dt.days.fillna(5))
    )
    starters["short_rest"] = (starters["days_rest"] <= 4).astype(int)

    # Save
    out_path = cfg["pitcher_features"]
    starters.to_csv(out_path, index=False)
    data["pitcher_features"] = starters
    print(f"  Pitcher features: {len(starters):,} rows. Saved to {out_path}")
    return starters


# ── Auto-run ───────────────────────────────────────────────────────────────────
build_pitcher_k_features(cfg)

Building pitcher K features...
  Pitcher features: 12,236 rows. Saved to C:\Users\lmayn\Downloads\mlb-betting\K_Pro_System\data\pitcher_k_features.csv


,game_pk,pitcher,bf,game_date,starter_ks,starter_ip,p_throws,home_team,away_team,n_pitches,...,chase_pct_L10,velo_mean_L5,velo_trend_L5,fb_pct_L10,breaking_pct_L10,primary_whiff_rate_L10,avg_ip_L5,avg_bf_L5,days_rest,short_rest
0,634568,425794,23,2021-04-08,6,1.0,R,STL,MIL,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0
1,634541,425794,23,2021-04-14,7,1.0,R,STL,WSH,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0
2,634509,425794,27,2021-04-20,10,1.0,R,WSH,STL,27.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,0
3,634345,425794,31,2021-04-26,8,1.0,R,STL,PHI,31.0,...,0.251852,83.444498,0.036312,0.404522,0.296833,0.203571,1.0,24.333333,6.0,0
4,634292,425794,26,2021-05-03,5,1.0,R,STL,NYM,27.0,...,0.282639,83.064825,-0.441083,0.392101,0.335528,0.224107,1.0,26.000000,7.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12231,823974,808967,21,2026-03-26,6,1.0,R,LAD,AZ,22.0,...,0.439524,89.340211,0.137227,0.358297,0.212055,0.351414,1.0,25.800000,182.0,0
12232,776310,813349,21,2025-09-16,7,1.0,L,BOS,ATH,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0
12233,776157,813349,20,2025-09-27,7,1.0,L,BOS,DET,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,0
12234,824538,813349,21,2026-03-29,6,1.0,L,CIN,BOS,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,183.0,0


## Section 5 — Lineup Feature Join

In [6]:
def _get_today_lineups(target_date: str = None):
    """
    Pull today's lineups from MLB Stats API.
    Returns dict: {game_pk: {home_team, away_team, home_pitcher, away_pitcher,
                              home_lineup: [...], away_lineup: [...]}}
    """
    if target_date is None:
        target_date = str(date.today())

    url = (f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={target_date}"
           f"&gameType=R&hydrate=probablePitcher,lineups")
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        sched = resp.json()
    except Exception as e:
        print(f"[WARN] MLB schedule API failed: {e}")
        return {}

    games = {}
    for date_block in sched.get("dates", []):
        for g in date_block.get("games", []):
            game_pk  = g["gamePk"]
            home_tm  = g["teams"]["home"]["team"]["abbreviation"]
            away_tm  = g["teams"]["away"]["team"]["abbreviation"]

            home_pp = g["teams"]["home"].get("probablePitcher", {}).get("fullName", "")
            away_pp = g["teams"]["away"].get("probablePitcher", {}).get("fullName", "")

            home_lu = [p.get("fullName", "") for p in
                       g.get("lineups", {}).get("homePlayers", [])]
            away_lu = [p.get("fullName", "") for p in
                       g.get("lineups", {}).get("awayPlayers", [])]

            games[game_pk] = {
                "home_team":    home_tm, "away_team":    away_tm,
                "home_pitcher": home_pp, "away_pitcher": away_pp,
                "home_lineup":  home_lu, "away_lineup":  away_lu,
            }
    return games


def build_lineup_k_features(cfg: dict, target_date: str = None) -> pd.DataFrame:
    """
    Build opponent K vulnerability features from Statcast batter data.
    One row per (game_pk, pitching_team, batting_team).
    Stores in data['lineup_features'], saves to lineup_k_features.csv.
    """
    if "statcast" not in data or len(data["statcast"]) == 0:
        print("[WARN] Statcast not loaded. Run Section 1 first.")
        data["lineup_features"] = pd.DataFrame()
        return data["lineup_features"]

    sc = data["statcast"].copy()
    if target_date is None:
        target_date = str(date.today())

    print(f"Building lineup K features for {target_date}...")

    # PA-level rows with batter team
    pa = sc[sc["events"].notna()].copy()
    if "game_date" not in pa.columns:
        print("[WARN] game_date missing from Statcast.")
        data["lineup_features"] = pd.DataFrame()
        return data["lineup_features"]

    pa["game_date"] = pd.to_datetime(pa["game_date"], errors="coerce")

    # Batting team from inning_topbot + home/away team fields
    if "inning_topbot" in pa.columns and "home_team" in pa.columns:
        pa["bat_team"] = np.where(pa["inning_topbot"] == "Bot",
                                   pa.get("home_team", ""),
                                   pa.get("away_team", ""))
    elif "bat_team" in pa.columns:
        pass
    else:
        print("[WARN] Cannot determine batting team. Lineup features will be empty.")
        data["lineup_features"] = pd.DataFrame()
        return data["lineup_features"]

    # K/swing/whiff/chase flags per PA row
    pa["is_k"]      = (pa["events"] == "strikeout").astype(int)
    pa["is_swing"]  = pa.get("description", pd.Series(dtype=str)).isin(
        ["swinging_strike", "swinging_strike_blocked", "foul", "foul_tip", "hit_into_play"]
    ).astype(int) if "description" in pa.columns else 0
    pa["is_whiff"]  = pa.get("description", pd.Series(dtype=str)).isin(
        ["swinging_strike", "swinging_strike_blocked"]
    ).astype(int) if "description" in pa.columns else 0
    pa["is_out_zone"] = (pa["zone"] > 9).astype(int) if "zone" in pa.columns else 0
    pa["is_chase"]    = (pa["is_out_zone"] & pa["is_swing"]).astype(int)

    # LHB flag
    if "stand" in pa.columns:
        pa["is_lhb"] = (pa["stand"] == "L").astype(int)
    else:
        pa["is_lhb"] = 0

    # Batting order slot (if available)
    if "bat_order" not in pa.columns:
        pa["bat_order"] = np.nan

    # Windowed date filter: L14 days from target_date
    td    = pd.Timestamp(target_date)
    pa_14 = pa[pa["game_date"] >= (td - timedelta(days=14))].copy()
    pa_50 = pa[pa["game_date"] >= (td - timedelta(days=50))].copy()

    # Today's lineups for game_pk mapping
    today_games = _get_today_lineups(target_date)

    if not today_games:
        print("[INFO] No lineup data from MLB API. Using team-level averages as fallback.")

    rows = []
    # Build features per team (opponent of each pitcher)
    for game_pk, g_info in today_games.items():
        for side in ["home", "away"]:
            # The pitcher on this side faces the opposite team's lineup
            pitcher_name  = g_info[f"{side}_pitcher"]
            bat_team      = g_info["away_team"] if side == "home" else g_info["home_team"]
            pitch_team    = g_info[f"{side}_team"]

            # Pitcher handedness from pitcher_features
            p_throws = "R"
            if "pitcher_features" in data and len(data["pitcher_features"]) > 0:
                pf = data["pitcher_features"]
                match = pf[pf["game_date"] == pf[pf["game_date"].notna()]["game_date"].max()]
                ph_row = pf[pf["game_date"] <= pd.Timestamp(target_date)]
                if len(ph_row) > 0 and "p_throws" in ph_row.columns:
                    p_throws = ph_row["p_throws"].iloc[-1]

            # L14 team stats
            team_14 = pa_14[pa_14["bat_team"] == bat_team]
            n14 = max(len(team_14), 1)

            opp_k_rate_L14        = team_14["is_k"].sum() / n14
            opp_chase_rate_L14    = team_14["is_chase"].sum() / n14
            opp_whiff_rate_L14    = team_14["is_whiff"].sum() / n14
            opp_lineup_pct_L      = team_14["is_lhb"].sum() / n14

            # K rate vs same pitcher hand
            team_14_hand = team_14[team_14.get("p_throws", pd.Series(dtype=str)) == p_throws] \
                           if "p_throws" in team_14.columns else team_14
            n14h = max(len(team_14_hand), 1)
            opp_k_rate_vs_hand_L14 = team_14_hand["is_k"].sum() / n14h

            # Top-3 slot K rate (L50)
            team_50 = pa_50[pa_50["bat_team"] == bat_team]
            if "bat_order" in team_50.columns and team_50["bat_order"].notna().any():
                top3 = team_50[team_50["bat_order"].isin([1, 2, 3])]
                weights = {1: 3, 2: 2, 3: 1}
                if len(top3) > 0:
                    top3 = top3.copy()
                    top3["w"] = top3["bat_order"].map(weights).fillna(1)
                    opp_top3_k_rate_L50 = (top3["is_k"] * top3["w"]).sum() / top3["w"].sum()
                else:
                    opp_top3_k_rate_L50 = opp_k_rate_L14
            else:
                opp_top3_k_rate_L50 = opp_k_rate_L14

            # Platoon K edge
            k_pct_L10 = np.nan
            if "pitcher_features" in data and len(data["pitcher_features"]) > 0:
                pf = data["pitcher_features"]
                pf_recent = pf[pf["game_date"] <= pd.Timestamp(target_date)]
                if len(pf_recent) > 0 and "k_pct_L10" in pf_recent.columns:
                    k_pct_L10 = pf_recent["k_pct_L10"].iloc[-1]

            opp_platoon_k_edge = (opp_lineup_pct_L - 0.5) * (k_pct_L10 if not np.isnan(k_pct_L10) else 0.2)

            rows.append({
                "game_pk":               game_pk,
                "game_date":             target_date,
                "pitcher_name":          pitcher_name,
                "pitch_team":            pitch_team,
                "bat_team":              bat_team,
                "opp_k_rate_L14":        opp_k_rate_L14,
                "opp_k_rate_vs_hand_L14":opp_k_rate_vs_hand_L14,
                "opp_chase_rate_L14":    opp_chase_rate_L14,
                "opp_whiff_rate_L14":    opp_whiff_rate_L14,
                "opp_lineup_pct_L":      opp_lineup_pct_L,
                "opp_platoon_k_edge":    opp_platoon_k_edge,
                "opp_top3_k_rate_L50":   opp_top3_k_rate_L50,
            })

    df = pd.DataFrame(rows)
    out_path = cfg["lineup_features"]
    df.to_csv(out_path, index=False)
    data["lineup_features"] = df
    print(f"  Lineup features: {len(df)} rows. Saved to {out_path}")
    return df


# ── Auto-run ───────────────────────────────────────────────────────────────────
build_lineup_k_features(cfg)

Building lineup K features for 2026-04-08...


KeyError: 'abbreviation'

## Section 6 — Feature Join Layer

In [ ]:
def _get_weather(team: str, target_date: str) -> tuple:
    """
    Get temperature_f and is_dome for a team's home stadium.
    Uses Open-Meteo API (free, no key). Returns (temperature_f, is_dome).
    """
    is_dome = int(team in DOME_TEAMS)
    if is_dome:
        return 70.0, 1  # dome: fixed 70°F, prevents air_density corruption

    coords = STADIUM_COORDS.get(team)
    if coords is None:
        return 70.0, 0

    lat, lon = coords
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude={lat}&longitude={lon}&hourly=temperature_2m"
           f"&temperature_unit=fahrenheit&timezone=America%2FNew_York"
           f"&forecast_days=1&start_date={target_date}&end_date={target_date}")
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        j = resp.json()
        temps = j.get("hourly", {}).get("temperature_2m", [])
        # Use hour index 13 (1pm ET) as game time proxy
        t = temps[13] if len(temps) > 13 else (temps[0] if temps else 70.0)
        return float(t), 0
    except Exception:
        return 70.0, 0


def build_feature_table(cfg: dict, target_date: str = None) -> pd.DataFrame:
    """
    Join pitcher features + lineup features + umpire features + weather context.
    Produces model_features.csv with one row per (starter, game_pk) and all K_FEATURES columns.
    Missing features filled with np.nan — XGBoost handles NaN natively.
    Stores in data['features'], saves to model_features.csv.
    """
    if target_date is None:
        target_date = str(date.today())

    print(f"Building feature table for {target_date}...")

    # Load pitcher features
    pf_path = cfg["pitcher_features"]
    if Path(pf_path).exists():
        pf = pd.read_csv(pf_path, low_memory=False)
        pf["game_date"] = pd.to_datetime(pf["game_date"], errors="coerce")
    elif "pitcher_features" in data and len(data["pitcher_features"]) > 0:
        pf = data["pitcher_features"].copy()
    else:
        print("[WARN] Pitcher features not found. Run Section 4.")
        data["features"] = pd.DataFrame()
        return data["features"]

    # Load lineup features
    lf_path = cfg["lineup_features"]
    if Path(lf_path).exists():
        lf = pd.read_csv(lf_path, low_memory=False)
    elif "lineup_features" in data and len(data.get("lineup_features", [])) > 0:
        lf = data["lineup_features"].copy()
    else:
        lf = pd.DataFrame()

    # Filter pitcher features to target_date
    td = pd.Timestamp(target_date)
    pf_today = pf[pf["game_date"] == td].copy()

    if len(pf_today) == 0:
        print(f"[INFO] No pitcher features for {target_date}. Using most recent available.")
        pf_today = pf[pf["game_date"] == pf["game_date"].max()].copy()

    # Merge lineup features
    if len(lf) > 0 and "game_pk" in lf.columns:
        lf_today = lf[lf.get("game_date", pd.Series(dtype=str)) == target_date] \
                   if "game_date" in lf.columns else lf
        pf_today = pf_today.merge(lf_today, on="game_pk", how="left")
    else:
        # Fallback: fill opponent features with historical team averages
        opp_cols = ["opp_k_rate_L14", "opp_k_rate_vs_hand_L14", "opp_chase_rate_L14",
                    "opp_whiff_rate_L14", "opp_lineup_pct_L", "opp_platoon_k_edge",
                    "opp_top3_k_rate_L50"]
        for col in opp_cols:
            pf_today[col] = np.nan

    # Merge umpire features on game_pk
    ump_cols = ["ump_overall_accuracy_L30", "ump_k_boost_L30", "ump_consistency_L30"]
    if "umps" in data and len(data["umps"]) > 0:
        umps = data["umps"]
        ump_today = umps[umps["date"] == pd.Timestamp(target_date)] if "date" in umps.columns else umps
        if "game_pk" in ump_today.columns and "game_pk" in pf_today.columns:
            pf_today = pf_today.merge(
                ump_today[["game_pk"] + ump_cols].drop_duplicates("game_pk"),
                on="game_pk", how="left"
            )
        else:
            # No game_pk in umpire — use mean values
            for col in ump_cols:
                if col in umps.columns:
                    pf_today[col] = umps[col].mean()
                else:
                    pf_today[col] = np.nan
    else:
        for col in ump_cols:
            pf_today[col] = np.nan

    # Weather context
    if "home_team" in pf_today.columns:
        weather_rows = pf_today["home_team"].apply(
            lambda t: pd.Series(_get_weather(str(t), target_date),
                                 index=["temperature_f", "is_dome"])
        )
        pf_today["temperature_f"] = weather_rows["temperature_f"]
        pf_today["is_dome"]       = weather_rows["is_dome"]
    else:
        pf_today["temperature_f"] = 70.0
        pf_today["is_dome"]       = 0

    # is_home flag
    if "home_team" in pf_today.columns and "pitch_team" in pf_today.columns:
        pf_today["is_home"] = (pf_today["pitch_team"] == pf_today["home_team"]).astype(int)
    else:
        pf_today["is_home"] = 0

    # implied_win_pct placeholder — overridden at signal generation
    pf_today["implied_win_pct"] = 0.5

    # Ensure all K_FEATURES columns exist (fill missing with np.nan)
    for col in K_FEATURES:
        if col not in pf_today.columns:
            pf_today[col] = np.nan

    # Save
    out_path = cfg["model_features"]
    pf_today.to_csv(out_path, index=False)
    data["features"] = pf_today
    print(f"  Feature table: {len(pf_today)} rows, {len(pf_today.columns)} cols. Saved to {out_path}")
    print(f"  K_FEATURES coverage: {sum(1 for f in K_FEATURES if f in pf_today.columns)}/{len(K_FEATURES)}")
    return pf_today


# ── Auto-run ───────────────────────────────────────────────────────────────────
build_feature_table(cfg)

## Section 7 — Model Train (OOS + Walk-forward CV)

**MANUAL GATE** — `TRAIN_ENABLED = False`. Do not run on kernel restart.

In [ ]:
TRAIN_ENABLED = False  # ← MANUAL GATE. Set True only when ready to retrain.

if not TRAIN_ENABLED:
    print("[GATE] Section 7 skipped. Set TRAIN_ENABLED = True to run.")
else:
    import warnings
    warnings.filterwarnings('ignore')

    def _mae(y_true, y_pred):
        return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

    def _rmse(y_true, y_pred):
        return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2)))

    def _r2(y_true, y_pred):
        ss_res = np.sum((np.array(y_true) - np.array(y_pred))**2)
        ss_tot = np.sum((np.array(y_true) - np.mean(y_true))**2)
        return 1 - ss_res / max(ss_tot, 1e-9)

    def _train_xgb(X_tr, y_tr, X_te, y_te, num_rounds=2000, early_stop=50):
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dtest  = xgb.DMatrix(X_te, label=y_te)
        params = XGB_PARAMS.copy()
        bst = xgb.train(
            params, dtrain,
            num_boost_round=num_rounds,
            evals=[(dtest, "test")],
            early_stopping_rounds=early_stop,
            verbose_eval=False,
        )
        return bst

    # Load features
    feat_path = cfg["model_features"]
    if not Path(feat_path).exists():
        print("[ERROR] model_features.csv not found. Run Sections 4-6 first.")
    else:
        df_all = pd.read_csv(feat_path, low_memory=False)
        df_all["game_date"] = pd.to_datetime(df_all["game_date"], errors="coerce")
        df_all["year"]      = df_all["game_date"].dt.year
        df_all = df_all.dropna(subset=["starter_ks"]).copy()
        df_all["starter_ks"] = pd.to_numeric(df_all["starter_ks"], errors="coerce")
        df_all = df_all.dropna(subset=["starter_ks"]).copy()

        # ── Walk-forward CV ───────────────────────────────────────────────────
        # Folds: train on prior 2 years, test on held-out year
        cv_folds = cfg["cv_folds"]  # [2023, 2024, 2025]
        wf_results = []

        print("\n=== Walk-forward CV ===")
        for test_year in cv_folds:
            train_years = [test_year - 2, test_year - 1]
            df_tr = df_all[df_all["year"].isin(train_years)].copy()
            df_te = df_all[df_all["year"] == test_year].copy()

            if len(df_tr) < 50 or len(df_te) < 10:
                print(f"  Fold {test_year}: insufficient data (train={len(df_tr)}, test={len(df_te)}). Skip.")
                continue

            X_tr = df_tr[K_FEATURES].fillna(0)
            y_tr = df_tr["starter_ks"]
            X_te = df_te[K_FEATURES].fillna(0)
            y_te = df_te["starter_ks"]

            bst_fold = _train_xgb(X_tr, y_tr, X_te, y_te)
            y_pred   = bst_fold.predict(xgb.DMatrix(X_te))

            fold_mae = _mae(y_te, y_pred)
            fold_rmse = _rmse(y_te, y_pred)
            fold_r2   = _r2(y_te, y_pred)
            fold_cal  = float(np.mean(y_pred)) - float(np.mean(y_te))
            best_it   = bst_fold.best_iteration

            wf_results.append({
                "test_year": test_year, "n_train": len(df_tr), "n_test": len(df_te),
                "mae": fold_mae, "rmse": fold_rmse, "r2": fold_r2,
                "cal_gap": fold_cal, "best_iteration": best_it
            })
            print(f"  Fold {test_year}: MAE={fold_mae:.3f}  RMSE={fold_rmse:.3f}  "
                  f"R²={fold_r2:.3f}  cal={fold_cal:+.3f}  "
                  f"n_train={len(df_tr)}  n_test={len(df_te)}  best_iter={best_it}")

        if wf_results:
            wf_df    = pd.DataFrame(wf_results)
            wf_mae   = float(wf_df["mae"].mean())
            wf_rmse  = float(wf_df["rmse"].mean())
            wf_r2    = float(wf_df["r2"].mean())
            wf_cal   = float(wf_df["cal_gap"].mean())
            wf_mae_s = float(wf_df["mae"].std())
            print(f"\n  WF mean MAE: {wf_mae:.3f} (std {wf_mae_s:.3f})  "
                  f"RMSE: {wf_rmse:.3f}  R²: {wf_r2:.3f}  cal: {wf_cal:+.3f}")
        else:
            wf_mae = wf_rmse = wf_r2 = wf_cal = wf_mae_s = np.nan

        # ── Post-CV OOS model: train on all except last fold, test on last ────
        print("\n=== OOS Model (train excl. last fold, test on last fold) ===")
        last_test_year = cv_folds[-1]
        df_tr_oos = df_all[df_all["year"] < last_test_year].copy()
        df_te_oos = df_all[df_all["year"] == last_test_year].copy()

        X_tr_oos = df_tr_oos[K_FEATURES].fillna(0)
        y_tr_oos = df_tr_oos["starter_ks"]
        X_te_oos = df_te_oos[K_FEATURES].fillna(0)
        y_te_oos = df_te_oos["starter_ks"]

        bst_oos   = _train_xgb(X_tr_oos, y_tr_oos, X_te_oos, y_te_oos)
        y_pred_oos = bst_oos.predict(xgb.DMatrix(X_te_oos))
        y_pred_tr  = bst_oos.predict(xgb.DMatrix(X_tr_oos))

        mae_oos    = _mae(y_te_oos, y_pred_oos)
        mae_train  = _mae(y_tr_oos, y_pred_tr)
        rmse_oos   = _rmse(y_te_oos, y_pred_oos)
        r2_oos     = _r2(y_te_oos, y_pred_oos)
        cal_oos    = float(np.mean(y_pred_oos)) - float(np.mean(y_te_oos))
        best_it    = bst_oos.best_iteration

        print(f"  OOS MAE={mae_oos:.3f}  RMSE={rmse_oos:.3f}  R²={r2_oos:.3f}  "
              f"cal={cal_oos:+.3f}  train_mae={mae_train:.3f}  "
              f"overfit_gap={abs(mae_train-mae_oos):.3f}  best_iter={best_it}")

        # Save OOS model
        bst_oos.save_model(cfg["model_oos"])
        print(f"  OOS model saved → {cfg['model_oos']}")

        # ── Meta JSON — save OOS metrics, never overwrite in 7b ───────────────
        meta = {
            "mae_oos":        mae_oos,
            "rmse_oos":       rmse_oos,
            "r2_oos":         r2_oos,
            "cal_oos":        cal_oos,
            "overfit_gap":    abs(mae_train - mae_oos),
            "wf_mae":         wf_mae,
            "wf_rmse":        wf_rmse,
            "wf_r2":          wf_r2,
            "wf_cal":         wf_cal,
            "wf_mae_std":     wf_mae_s,
            "cv_folds":       cv_folds,
            "best_iteration": best_it,
            "features":       K_FEATURES,
            "n_features":     len(K_FEATURES),
            "trained_at":     datetime.now().isoformat(),
            "n_train_oos":    len(df_tr_oos),
            "n_test_oos":     len(df_te_oos),
        }
        with open(cfg["model_meta"], "w") as f:
            json.dump(meta, f, indent=2)
        print(f"  Meta saved → {cfg['model_meta']}")

        # Store in runtime
        model["bst"]      = bst_oos
        model["features"] = K_FEATURES
        print("\n  WF mean MAE is the headline metric. OOS is for validation only.")

## Section 7b — Full Retrain (Production)

**MANUAL GATE** — `FULL_RETRAIN_ENABLED = False`. Pre-season only.
Uses `best_iteration * 1.1` from OOS meta. Never overwrites OOS evaluation metrics.

In [ ]:
FULL_RETRAIN_ENABLED = False  # ← MANUAL GATE. Pre-season only.

if not FULL_RETRAIN_ENABLED:
    print("[GATE] Section 7b skipped. Set FULL_RETRAIN_ENABLED = True to run.")
else:
    # Load meta — must exist from Section 7
    meta_path = cfg["model_meta"]
    if not Path(meta_path).exists():
        print("[ERROR] model_meta not found. Run Section 7 first.")
    else:
        with open(meta_path, "r") as f:
            meta = json.load(f)

        best_it = meta.get("best_iteration", 500)
        num_rounds = int(best_it * 1.1)
        print(f"Full retrain: {num_rounds} rounds (best_iter={best_it} x 1.1)")

        feat_path = cfg["model_features"]
        df_all = pd.read_csv(feat_path, low_memory=False)
        df_all["game_date"] = pd.to_datetime(df_all["game_date"], errors="coerce")
        df_all = df_all.dropna(subset=["starter_ks"]).copy()
        df_all["starter_ks"] = pd.to_numeric(df_all["starter_ks"], errors="coerce")
        df_all = df_all.dropna(subset=["starter_ks"]).copy()

        X_all = df_all[K_FEATURES].fillna(0)
        y_all = df_all["starter_ks"]

        dtrain = xgb.DMatrix(X_all, label=y_all)
        bst_prod = xgb.train(
            XGB_PARAMS, dtrain,
            num_boost_round=num_rounds,
            verbose_eval=False,
        )
        bst_prod.save_model(cfg["model_prod"])
        print(f"Production model saved → {cfg['model_prod']}")

        # Update meta — SAFE KEYS ONLY. Never touch OOS evaluation metrics.
        IMMUTABLE_KEYS = {
            "mae_oos", "rmse_oos", "r2_oos", "cal_oos", "overfit_gap",
            "wf_mae", "wf_rmse", "wf_r2", "wf_cal", "wf_mae_std",
            "cv_folds", "best_iteration"
        }
        meta["trained_at"]   = datetime.now().isoformat()
        meta["train_rows"]   = len(df_all)
        meta["train_through"]= str(df_all["game_date"].max().date())
        meta["full_retrain"] = True
        meta["features"]     = K_FEATURES
        meta["n_features"]   = len(K_FEATURES)

        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)
        print(f"Meta updated (safe keys only) → {meta_path}")

        model["bst"]      = bst_prod
        model["features"] = K_FEATURES
        print(f"Production model loaded into runtime model store.")

## Section 8 — Monte Carlo K Distribution Engine

In [ ]:
def _load_model(cfg: dict) -> bool:
    """
    Load production model first, fallback to OOS model.
    Stores in model['bst'] and model['features'].
    Returns True if successful.
    """
    for path_key in ["model_prod", "model_oos"]:
        path = cfg[path_key]
        if Path(path).exists():
            bst = xgb.Booster()
            bst.load_model(path)
            model["bst"]      = bst
            model["features"] = K_FEATURES
            print(f"Model loaded from {path}")

            # Load meta for overdispersion check
            meta_path = cfg["model_meta"]
            if Path(meta_path).exists():
                with open(meta_path, "r") as f:
                    model["meta"] = json.load(f)

            return True
    print("[WARN] No trained model found. Run Section 7 to train.")
    return False


def _check_overdispersion(cfg: dict) -> tuple:
    """
    Check for overdispersion in predicted lambdas vs actuals.
    Returns (use_negbinom, nb_r, nb_p).
    """
    if "bst" not in model:
        return False, None, None

    feat_path = cfg["model_features"]
    if not Path(feat_path).exists():
        return False, None, None

    df = pd.read_csv(feat_path, low_memory=False)
    df = df.dropna(subset=["starter_ks"]).copy()
    if len(df) < 50:
        return False, None, None

    X = df[K_FEATURES].fillna(0)
    lambdas = model["bst"].predict(xgb.DMatrix(X))
    actuals = df["starter_ks"].values

    mean_l = float(np.mean(lambdas))
    var_l  = float(np.var(lambdas))

    use_nb = var_l > mean_l * 1.2
    nb_r, nb_p = None, None

    if use_nb:
        mean_k = float(np.mean(actuals))
        var_k  = float(np.var(actuals))
        if var_k > mean_k and mean_k > 0:
            p_est = mean_k / var_k
            r_est = mean_k * p_est / (1 - p_est)
            nb_r, nb_p = max(r_est, 0.1), max(p_est, 0.01)

            # KS test on residuals
            residuals = actuals - lambdas
            ks_stat, ks_p = kstest(residuals, "norm",
                                    args=(residuals.mean(), residuals.std()))
            print(f"  Overdispersion detected → Negative Binomial. "
                  f"r={nb_r:.3f} p={nb_p:.3f} KS p={ks_p:.4f}")
        else:
            use_nb = False
    else:
        print(f"  Distribution: Poisson (var/mean ratio: {var_l/max(mean_l,1e-9):.3f})")

    return use_nb, nb_r, nb_p


def simulate_k_distribution(lambda_k: float, avg_ip_L5: float = None,
                              n_sims: int = 10_000, cap: int = 14,
                              use_negbinom: bool = False,
                              nb_r: float = None, nb_p: float = None) -> dict:
    """
    Monte Carlo K distribution.
    Applies early exit IP penalty if avg_ip_L5 < 5.0.
    Draws from Poisson or Negative Binomial.
    Hard cap at cap=14.
    Returns dict with lambda_k, mean, median, p10, p90, dist, p_1plus..p_14plus.
    """
    # Early exit IP penalty
    if avg_ip_L5 is not None and not np.isnan(avg_ip_L5) and avg_ip_L5 < 5.0:
        ip_factor = avg_ip_L5 / 5.0
        lambda_k  = max(lambda_k * ip_factor, 0.5)

    lambda_k = max(lambda_k, 0.1)  # floor

    rng = np.random.default_rng(42)
    if use_negbinom and nb_r is not None and nb_p is not None:
        # Negative Binomial: n=r, p=p in scipy convention
        # Mean = r*(1-p)/p, so we need to re-parameterize to match lambda
        # Use r from fit, adjust p so mean = lambda_k
        p_adj = nb_r / (nb_r + lambda_k)
        samples = nbinom.rvs(nb_r, p_adj, size=n_sims, random_state=rng.integers(0, 2**31))
        dist_type = "negbinom"
    else:
        samples = rng.poisson(lambda_k, size=n_sims)
        dist_type = "poisson"

    samples = np.clip(samples, 0, cap)

    result = {
        "lambda_k":  lambda_k,
        "mean":      float(np.mean(samples)),
        "median":    float(np.median(samples)),
        "p10":       float(np.percentile(samples, 10)),
        "p90":       float(np.percentile(samples, 90)),
        "dist":      dist_type,
    }
    for n in range(1, cap + 1):
        result[f"p_{n}plus"] = float(np.mean(samples >= n))

    return result


def ou_edge(probs: dict, k_line: float, over_odds: int, under_odds: int) -> dict:
    """
    Compute O/U edge vs DK line.
    k_line=5.5 → over threshold = 6+ Ks
    """
    rung    = int(math.ceil(k_line))
    rung    = max(1, min(rung, 14))
    p_over  = probs.get(f"p_{rung}plus", 0.0)
    p_under = 1.0 - p_over

    imp_over  = american_to_implied_prob(over_odds)
    imp_under = american_to_implied_prob(under_odds)
    fair_over, fair_under = remove_vig(imp_over, imp_under)

    return {
        "over_edge":   p_over  - fair_over,
        "under_edge":  p_under - fair_under,
        "p_over":      p_over,
        "p_under":     p_under,
        "fair_over":   fair_over,
        "fair_under":  fair_under,
        "rung":        rung,
    }


def ladder_edges(probs: dict, ldr_odds: dict) -> dict:
    """
    Compute edge for each ladder rung.
    ldr_odds: {"k_5plus": -200, "k_6plus": +150, ...}
    No vig removal on ladder (one-sided market).
    Returns {"k_5plus": {"edge": 0.08, "odds": -200, "p_model": 0.65}, ...}
    """
    results = {}
    for rung_key, odds in ldr_odds.items():
        if odds is None or not isinstance(odds, (int, float)):
            continue
        # Parse rung: "k_5plus" → 5
        m = re.match(r"k_(\d+)plus", rung_key)
        if not m:
            continue
        n = int(m.group(1))
        if n < 1 or n > 14:
            continue

        p_model  = probs.get(f"p_{n}plus", 0.0)
        mkt_p    = american_to_implied_prob(int(odds))
        edge     = p_model - mkt_p

        results[rung_key] = {
            "edge":    edge,
            "odds":    int(odds),
            "p_model": p_model,
            "mkt_p":   mkt_p,
            "n":       n,
        }
    return results


def _build_feature_vector(pitcher_name: str, target_date: str) -> pd.DataFrame:
    """
    Build a single-row feature vector for inference.
    Pulls from data['features'] if available, else median profile fallback.
    """
    feat_path = cfg["model_features"]
    if Path(feat_path).exists():
        df = pd.read_csv(feat_path, low_memory=False)
    elif "features" in data:
        df = data["features"]
    else:
        df = pd.DataFrame()

    if len(df) > 0 and "pitcher_name" in df.columns:
        norm_target = normalize_name(pitcher_name)
        df["_norm"] = df["pitcher_name"].apply(normalize_name)
        match = df[df["_norm"] == norm_target]
        if len(match) > 0:
            return match.tail(1)[K_FEATURES].fillna(0)

    # Fallback: median profile from all features
    if len(df) > 0:
        medians = df[K_FEATURES].median()
        row = pd.DataFrame([medians])
        return row

    return pd.DataFrame([{f: 0.0 for f in K_FEATURES}])


def generate_signals(cfg: dict, odds_df: pd.DataFrame = None,
                      target_date: str = None) -> pd.DataFrame:
    """
    Generate K prop signals for all pitchers in today's odds.
    Handles O/U and ladder bets independently.
    One row per pitcher-bet-type combination (always emit, even PASS).
    """
    if target_date is None:
        target_date = str(date.today())

    if "bst" not in model:
        loaded = _load_model(cfg)
        if not loaded:
            print("[ERROR] No model available. Train model first (Section 7).")
            return pd.DataFrame()

    use_nb, nb_r, nb_p = _check_overdispersion(cfg)

    if odds_df is None:
        if "k_odds" in data and len(data["k_odds"]) > 0:
            odds_df = data["k_odds"]
        elif Path(cfg["k_odds_master"]).exists():
            odds_df = pd.read_csv(cfg["k_odds_master"], low_memory=False)
            odds_df = odds_df[odds_df["date"] == target_date]
        else:
            print("[WARN] No odds available. Run Section 3 (DK scraper) first.")
            return pd.DataFrame()

    bankroll = cfg["bankroll"]
    rows     = []

    for _, pitcher_row in odds_df.iterrows():
        pitcher_name = pitcher_row.get("pitcher_name", "")
        if not pitcher_name:
            continue

        team     = pitcher_row.get("team", "")
        opp      = pitcher_row.get("opponent", "")
        k_line   = pitcher_row.get("k_line")
        ov_odds  = pitcher_row.get("over_odds")
        un_odds  = pitcher_row.get("under_odds")

        # Get feature vector
        fvec = _build_feature_vector(pitcher_name, target_date)

        # Override weather from today's context
        is_dome = int(team in DOME_TEAMS)
        temp_f, _ = _get_weather(str(team), target_date)
        fvec = fvec.copy()
        fvec["temperature_f"] = temp_f
        fvec["is_dome"]       = is_dome

        # Predict lambda
        X = fvec[K_FEATURES].fillna(0)
        lambda_k = float(model["bst"].predict(xgb.DMatrix(X))[0])

        # avg_ip_L5 for early exit penalty
        avg_ip = float(fvec["avg_ip_L5"].iloc[0]) if "avg_ip_L5" in fvec.columns \
                 and not pd.isna(fvec["avg_ip_L5"].iloc[0]) else None

        # Monte Carlo
        probs = simulate_k_distribution(
            lambda_k, avg_ip_L5=avg_ip,
            n_sims=cfg["mc_sims"], cap=cfg["mc_cap"],
            use_negbinom=use_nb, nb_r=nb_r, nb_p=nb_p
        )
        proj_k      = probs["mean"]
        lambda_adj  = probs["lambda_k"]
        dist_type   = probs["dist"]

        # ── O/U signal ────────────────────────────────────────────────────────
        if k_line is not None and ov_odds is not None and un_odds is not None:
            try:
                k_line   = float(k_line)
                ov_odds  = int(ov_odds)
                un_odds  = int(un_odds)
                ou_res   = ou_edge(probs, k_line, ov_odds, un_odds)

                for side, edge_val, odds_val, prob_val in [
                    ("ou_over",  ou_res["over_edge"],  ov_odds, ou_res["p_over"]),
                    ("ou_under", ou_res["under_edge"], un_odds, ou_res["p_under"]),
                ]:
                    kpct  = kelly_pct(edge_val, odds_val, cfg["kelly_fraction"])
                    stake = kelly_stake(edge_val, odds_val, bankroll,
                                        cfg["kelly_fraction"],
                                        cfg["min_kelly_pct"], cfg["max_kelly_pct"])
                    signal = ("BET" if kpct >= cfg["min_kelly_pct"]
                              and edge_val >= cfg["min_edge"] else "PASS")

                    rows.append({
                        "pitcher": pitcher_name, "team": team, "opp": opp,
                        "bet_type": side, "k_line": k_line, "odds": odds_val,
                        "proj_k": proj_k, "lambda_k": lambda_adj,
                        "edge": edge_val, "kelly_pct": kpct, "stake": stake,
                        "signal": signal, "dist": dist_type,
                    })
            except (TypeError, ValueError):
                pass

        # ── Ladder signals ────────────────────────────────────────────────────
        ldr_odds = {}
        for n in range(1, 15):
            v = pitcher_row.get(f"k_{n}plus")
            if v is not None and not (isinstance(v, float) and np.isnan(v)):
                try:
                    ldr_odds[f"k_{n}plus"] = int(v)
                except (TypeError, ValueError):
                    pass

        if ldr_odds:
            ldr_res     = ladder_edges(probs, ldr_odds)
            n_corr      = 0   # count of ladder bets placed for this pitcher
            for rung_key, rinfo in sorted(ldr_res.items(),
                                           key=lambda x: x[1]["n"]):
                edge_val  = rinfo["edge"]
                odds_val  = rinfo["odds"]

                # Correlated Kelly scaling
                corr_scale = 1.0 / math.sqrt(max(n_corr, 1))
                kpct  = kelly_pct(edge_val, odds_val, cfg["kelly_fraction"]) * corr_scale
                stake = kelly_stake(edge_val, odds_val, bankroll,
                                    cfg["kelly_fraction"],
                                    cfg["min_kelly_pct"], cfg["max_kelly_pct"]) * corr_scale

                signal = ("BET" if kpct >= cfg["min_kelly_pct"]
                          and edge_val >= cfg["min_edge"]
                          and n_corr < cfg["max_ladder_rungs"] else "PASS")

                if signal == "BET":
                    n_corr += 1

                rows.append({
                    "pitcher": pitcher_name, "team": team, "opp": opp,
                    "bet_type": rung_key, "k_line": rinfo["n"], "odds": odds_val,
                    "proj_k": proj_k, "lambda_k": lambda_adj,
                    "edge": edge_val, "kelly_pct": kpct, "stake": stake,
                    "signal": signal, "dist": dist_type,
                })

        # If no O/U or ladder, still emit a PASS row
        if not rows or rows[-1]["pitcher"] != pitcher_name:
            rows.append({
                "pitcher": pitcher_name, "team": team, "opp": opp,
                "bet_type": "pass", "k_line": None, "odds": None,
                "proj_k": proj_k, "lambda_k": lambda_adj,
                "edge": None, "kelly_pct": None, "stake": 0,
                "signal": "PASS", "dist": dist_type,
            })

    return pd.DataFrame(rows)


# Attempt to load model at section run time
_load_model(cfg)
print("Section 8 loaded.")

## Section 9 — Mathematical Rigor Assessment

In [ ]:
def run_rigor_assessment(cfg: dict):
    """
    Run all pre-production rigor checks against OOS meta.
    Prints PASS/FAIL per check.
    All must pass before promoting to live betting.
    """
    meta_path = cfg["model_meta"]
    feat_path = cfg["model_features"]

    if not Path(meta_path).exists():
        print("[ERROR] model_meta not found. Run Section 7 first.")
        return

    with open(meta_path, "r") as f:
        meta = json.load(f)

    print("\n" + "="*60)
    print("  K PRO v1 — MATHEMATICAL RIGOR ASSESSMENT")
    print("="*60)

    checks = []

    def _check(name, passed, actual, target, fmt=".3f"):
        status = "PASS" if passed else "FAIL"
        print(f"  {status:4s}  {name:40s}  actual={actual:{fmt}}  target={target}")
        checks.append(passed)

    mae_oos   = meta.get("mae_oos", float("inf"))
    r2_oos    = meta.get("r2_oos", -float("inf"))
    cal_oos   = meta.get("cal_oos", float("inf"))
    wf_mae    = meta.get("wf_mae",  float("inf"))
    wf_mae_s  = meta.get("wf_mae_std", float("inf"))
    overfit   = meta.get("overfit_gap", float("inf"))

    _check("MAE < 1.2 Ks",         mae_oos < 1.2,             mae_oos,  "< 1.2")
    _check("R² > 0.15",            r2_oos > 0.15,             r2_oos,   "> 0.15")
    _check("Calibration within 5%",abs(cal_oos) < 0.3,        cal_oos,  "< 0.05 * mean_actual")
    _check("Temporal stability",   wf_mae_s < 0.3,            wf_mae_s, "MAE std < 0.3")
    _check("Overfit gap < 0.3",    overfit < 0.3,             overfit,  "< 0.3")

    # Per-year MAE breakdown from features CSV
    if Path(feat_path).exists() and "bst" in model:
        df = pd.read_csv(feat_path, low_memory=False)
        df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
        df["year"] = df["game_date"].dt.year
        df = df.dropna(subset=["starter_ks"]).copy()

        print("\n  Per-year MAE breakdown:")
        for yr in sorted(df["year"].unique()):
            sub = df[df["year"] == yr]
            X = sub[K_FEATURES].fillna(0)
            y_pred = model["bst"].predict(xgb.DMatrix(X))
            y_true = sub["starter_ks"].values
            yr_mae = float(np.mean(np.abs(y_true - y_pred)))
            print(f"    {yr}:  MAE={yr_mae:.3f}  n={len(sub)}")

        # Leakage check: no single feature R² > 0.25
        print("\n  Leakage check (univariate R² per feature):")
        y_all = df["starter_ks"].values
        leak_fail = []
        for feat in K_FEATURES:
            if feat in df.columns:
                x_f = df[feat].fillna(df[feat].median()).values
                if x_f.std() > 0:
                    corr = np.corrcoef(x_f, y_all)[0, 1]
                    r2_f = corr ** 2
                    if r2_f > 0.20:
                        print(f"    [WARN] {feat}: R²={r2_f:.3f}")
                    if r2_f > 0.25:
                        leak_fail.append(feat)

        leakage_ok = len(leak_fail) == 0
        _check("Leakage: no feature R² > 0.25", leakage_ok,
               len(leak_fail), "0 failing features", "d")

        # KS test on residuals
        X_all   = df[K_FEATURES].fillna(0)
        y_pred_all = model["bst"].predict(xgb.DMatrix(X_all))
        residuals  = df["starter_ks"].values - y_pred_all
        ks_stat, ks_p = kstest(residuals, "norm",
                                args=(residuals.mean(), residuals.std()))
        _check("KS p > 0.05 (residuals not non-normal)", ks_p > 0.05,
               ks_p, "> 0.05")

    n_pass = sum(checks)
    n_fail = len(checks) - n_pass
    print("\n" + "="*60)
    print(f"  {n_pass}/{len(checks)} checks PASSED  |  {n_fail} FAILED")
    if n_fail > 0:
        print("  *** NOT ready for production. Fix failing checks first. ***")
    else:
        print("  All checks passed. Review WF MAE before going live.")
    print("="*60)


# run_rigor_assessment(cfg)

## Section 10 — Backtest Engine

In [ ]:
def run_backtest(cfg: dict, start_date: str = None, end_date: str = None) -> pd.DataFrame:
    """
    Simulate O/U bets at Kelly sizing on historical odds + actuals.
    Requires k_odds_master.csv with historical rows AND model_features.csv with starter_ks actuals.
    Prints: n bets, win rate, ROI, final bankroll.
    """
    if "bst" not in model:
        if not _load_model(cfg):
            print("[ERROR] No model available.")
            return pd.DataFrame()

    odds_path = cfg["k_odds_master"]
    feat_path = cfg["model_features"]

    if not Path(odds_path).exists():
        print(f"[WARN] k_odds_master.csv not found: {odds_path}")
        return pd.DataFrame()
    if not Path(feat_path).exists():
        print(f"[WARN] model_features.csv not found: {feat_path}")
        return pd.DataFrame()

    odds_df = pd.read_csv(odds_path, low_memory=False)
    feat_df = pd.read_csv(feat_path, low_memory=False)
    feat_df["game_date"] = pd.to_datetime(feat_df["game_date"], errors="coerce")

    if start_date:
        odds_df = odds_df[odds_df["date"] >= start_date]
    if end_date:
        odds_df = odds_df[odds_df["date"] <= end_date]

    if len(odds_df) == 0:
        print("[WARN] No odds rows in specified date range.")
        return pd.DataFrame()

    # Actuals lookup: (date, normalized_pitcher) → starter_ks
    if "starter_ks" in feat_df.columns and "pitcher_name" in feat_df.columns:
        feat_df["_norm"]  = feat_df["pitcher_name"].apply(normalize_name)
        feat_df["_date"]  = feat_df["game_date"].dt.strftime("%Y-%m-%d")
        actuals_map = feat_df.set_index(["_date", "_norm"])["starter_ks"].to_dict()
    else:
        actuals_map = {}

    bankroll = cfg["bankroll"]
    bet_log  = []

    for _, row in odds_df.iterrows():
        pitcher_name = row.get("pitcher_name", "")
        dt           = row.get("date", "")
        k_line       = row.get("k_line")
        ov_odds      = row.get("over_odds")
        un_odds      = row.get("under_odds")

        if not pitcher_name or k_line is None or ov_odds is None or un_odds is None:
            continue

        try:
            k_line  = float(k_line)
            ov_odds = int(ov_odds)
            un_odds = int(un_odds)
        except (TypeError, ValueError):
            continue

        # Get feature vector for this pitcher on this date
        fvec = _build_feature_vector(pitcher_name, str(dt))
        X    = fvec[K_FEATURES].fillna(0)
        lambda_k = float(model["bst"].predict(xgb.DMatrix(X))[0])

        avg_ip = float(fvec["avg_ip_L5"].iloc[0]) if "avg_ip_L5" in fvec.columns \
                  and not pd.isna(fvec["avg_ip_L5"].iloc[0]) else None

        probs  = simulate_k_distribution(lambda_k, avg_ip_L5=avg_ip,
                                          n_sims=cfg["mc_sims"], cap=cfg["mc_cap"])
        ou_res = ou_edge(probs, k_line, ov_odds, un_odds)

        actual = actuals_map.get((str(dt), normalize_name(pitcher_name)))

        for side, edge_val, odds_val in [
            ("ou_over",  ou_res["over_edge"],  ov_odds),
            ("ou_under", ou_res["under_edge"], un_odds),
        ]:
            stake = kelly_stake(edge_val, odds_val, bankroll,
                                cfg["kelly_fraction"],
                                cfg["min_kelly_pct"], cfg["max_kelly_pct"])
            if stake <= 0:
                continue

            if actual is not None:
                if side == "ou_over":
                    win = actual > k_line
                else:
                    win = actual < k_line

                if win:
                    if odds_val > 0:
                        pl = stake * odds_val / 100
                    else:
                        pl = stake * 100 / abs(odds_val)
                else:
                    pl = -stake

                bankroll += pl
                bet_log.append({
                    "date": dt, "pitcher": pitcher_name, "bet_type": side,
                    "k_line": k_line, "odds": odds_val, "stake": stake,
                    "edge": edge_val, "actual_ks": actual,
                    "win": int(win), "pl": pl, "bankroll": bankroll
                })

    df_log = pd.DataFrame(bet_log)
    if len(df_log) == 0:
        print("Backtest: No bets simulated (no matching actuals).")
        print("< 200 bets = statistically meaningless. Track CLV instead.")
        return df_log

    n_bets   = len(df_log)
    win_rate = df_log["win"].mean()
    roi      = df_log["pl"].sum() / df_log["stake"].sum()
    final_br = df_log["bankroll"].iloc[-1]

    print(f"\nBacktest Results:")
    print(f"  Bets:        {n_bets}")
    print(f"  Win rate:    {win_rate:.1%}")
    print(f"  ROI:         {roi:+.1%}")
    print(f"  Final BRL:   ${final_br:.2f}")
    if n_bets < 200:
        print("  WARNING: < 200 bets = statistically meaningless. Track CLV instead.")

    return df_log


# df_backtest = run_backtest(cfg)
print("Section 10 loaded. Call run_backtest(cfg) to run.")

## Section 11 — Bet Tracker (SQLite)

In [ ]:
def _init_db(cfg: dict):
    """Initialize SQLite database and create bets table if not exists."""
    con = sqlite3.connect(cfg["db_path"])
    con.execute("""
        CREATE TABLE IF NOT EXISTS bets (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            date           TEXT,
            pitcher_name   TEXT,
            team           TEXT,
            opponent       TEXT,
            bet_type       TEXT,
            k_line         REAL,
            odds           INTEGER,
            stake          REAL,
            lambda_k       REAL,
            proj_k         REAL,
            market_p       REAL,
            edge           REAL,
            kelly_pct      REAL,
            result         TEXT,
            actual_ks      INTEGER,
            profit_loss    REAL,
            created_at     TEXT,
            paper          INTEGER DEFAULT 1
        )
    """)
    con.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_dedup
        ON bets(date, pitcher_name, bet_type, k_line)
    """)
    con.commit()
    return con


def log_bets(signals_df: pd.DataFrame, cfg: dict):
    """
    Log all BET-signal rows to SQLite. Skips duplicates via INSERT OR IGNORE.
    """
    if len(signals_df) == 0:
        print("[INFO] No signals to log.")
        return

    bets = signals_df[signals_df["signal"] == "BET"].copy()
    if len(bets) == 0:
        print("[INFO] No BET signals. Nothing logged.")
        return

    con = _init_db(cfg)
    now = datetime.now().isoformat()
    logged = 0

    for _, row in bets.iterrows():
        odds     = row.get("odds")
        market_p = american_to_implied_prob(int(odds)) if odds is not None else None

        try:
            con.execute("""
                INSERT OR IGNORE INTO bets
                (date, pitcher_name, team, opponent, bet_type, k_line, odds,
                 stake, lambda_k, proj_k, market_p, edge, kelly_pct,
                 result, actual_ks, profit_loss, created_at, paper)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,NULL,NULL,NULL,?,?)
            """, (
                str(date.today()), row.get("pitcher", ""),
                row.get("team", ""), row.get("opp", ""),
                row.get("bet_type", ""), row.get("k_line"),
                row.get("odds"), row.get("stake"),
                row.get("lambda_k"), row.get("proj_k"), market_p,
                row.get("edge"), row.get("kelly_pct"),
                now, int(cfg["paper_mode"])
            ))
            logged += con.execute("SELECT changes()").fetchone()[0]
        except sqlite3.IntegrityError:
            pass  # Duplicate — already logged

    con.commit()
    con.close()
    print(f"  Logged {logged} new bets to {cfg['db_path']} (paper={cfg['paper_mode']})")


def settle_bets(cfg: dict, settle_date: str = None):
    """
    Settle pending bets from MLB Stats API boxscore.
    Defaults to yesterday. Matches pitcher by normalize_name().
    Settlement: ou_over → win if actual_ks > k_line
                ou_under → win if actual_ks < k_line
                k_Nplus → win if actual_ks >= N
    """
    if settle_date is None:
        settle_date = str(date.today() - timedelta(days=1))

    con = _init_db(cfg)

    # Get unsettled bets for settle_date
    pending = pd.read_sql(
        "SELECT * FROM bets WHERE date=? AND result IS NULL",
        con, params=(settle_date,)
    )

    if len(pending) == 0:
        print(f"[INFO] No unsettled bets for {settle_date}.")
        con.close()
        return

    print(f"Settling {len(pending)} bets for {settle_date}...")

    # Fetch schedule for settle_date to get game_pks
    url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={settle_date}&gameType=R"
    try:
        resp = requests.get(url, timeout=15)
        sched = resp.json()
    except Exception as e:
        print(f"[WARN] MLB API error: {e}")
        con.close()
        return

    game_pks = []
    for db in sched.get("dates", []):
        for g in db.get("games", []):
            game_pks.append(g["gamePk"])

    # Build actual Ks map: {normalized_pitcher_name: actual_ks}
    actuals = {}
    for gpk in game_pks:
        try:
            bs_url = f"https://statsapi.mlb.com/api/v1/game/{gpk}/boxscore"
            bsr = requests.get(bs_url, timeout=10).json()
            for side in ["home", "away"]:
                pitchers = bsr["teams"][side].get("pitchers", [])
                players  = bsr["teams"][side].get("players", {})
                if pitchers:
                    starter_id = pitchers[0]  # first entry = starter
                    pid_key    = f"ID{starter_id}"
                    p_stats = players.get(pid_key, {}).get("stats", {}).get("pitching", {})
                    k_count = p_stats.get("strikeOuts", None)
                    full_name = players.get(pid_key, {}).get("person", {}).get("fullName", "")
                    if full_name and k_count is not None:
                        actuals[normalize_name(full_name)] = int(k_count)
        except Exception:
            pass

    settled = 0
    for _, bet in pending.iterrows():
        norm_p = normalize_name(bet["pitcher_name"])
        actual = actuals.get(norm_p)

        if actual is None:
            # Fuzzy fallback: partial match
            for aname, aks in actuals.items():
                parts = norm_p.split()
                if parts and parts[-1] in aname:  # last name match
                    actual = aks
                    break

        if actual is None:
            continue

        bet_type = bet["bet_type"]
        k_line   = float(bet["k_line"])
        odds     = int(bet["odds"])
        stake    = float(bet["stake"])

        # Settlement logic
        if bet_type == "ou_over":
            win = actual > k_line
        elif bet_type == "ou_under":
            win = actual < k_line
        else:
            # Ladder: k_Nplus
            m = re.match(r"k_(\d+)plus", str(bet_type))
            if m:
                n   = int(m.group(1))
                win = actual >= n
            else:
                continue

        result = "win" if win else "loss"
        if win:
            pl = stake * (odds / 100 if odds > 0 else 100 / abs(odds))
        else:
            pl = -stake

        con.execute("""
            UPDATE bets SET result=?, actual_ks=?, profit_loss=?
            WHERE id=?
        """, (result, actual, round(pl, 2), int(bet["id"])))
        settled += 1

    con.commit()
    con.close()
    print(f"  Settled {settled}/{len(pending)} bets for {settle_date}.")


def bet_summary(cfg: dict):
    """Print bet tracker summary: settled count, W-L, ROI, P&L, progress to 200."""
    con = _init_db(cfg)
    df  = pd.read_sql("SELECT * FROM bets WHERE result IS NOT NULL", con)
    con.close()

    if len(df) == 0:
        print("No settled bets yet.")
        return

    n       = len(df)
    wins    = (df["result"] == "win").sum()
    losses  = (df["result"] == "loss").sum()
    total_pl = df["profit_loss"].sum()
    total_s  = df["stake"].sum()
    roi      = total_pl / max(total_s, 1)
    remaining = max(200 - n, 0)

    print("\n" + "="*50)
    print("  K PRO v1 — BET TRACKER SUMMARY")
    print("="*50)
    print(f"  Settled bets:  {n}  ({wins}W - {losses}L)")
    print(f"  Win rate:      {wins/max(n,1):.1%}")
    print(f"  ROI:           {roi:+.1%}")
    print(f"  P&L:           ${total_pl:+.2f}")
    print(f"  To gate (200): {remaining} bets remaining")
    print(f"  paper_mode:    {cfg['paper_mode']}")
    if remaining > 0:
        print(f"  *** Paper trading until {remaining} more bets settled. Track CLV. ***")
    print("="*50)


_init_db(cfg)
print(f"Section 11 loaded. DB initialized at {cfg['db_path']}")

## Section 12a — Quick Refresh

Re-scrape DK odds, generate signals, print dashboard, log bets.
**Run manually** — do not auto-execute.

In [ ]:
def print_dashboard(signals: pd.DataFrame, cfg: dict):
    """Print formatted K Pro signal dashboard."""
    today    = str(date.today())
    bankroll = cfg["bankroll"]
    paper    = cfg["paper_mode"]

    # Attempt to get settled count for running bankroll
    try:
        con = sqlite3.connect(cfg["db_path"])
        settled = pd.read_sql("SELECT profit_loss FROM bets WHERE result IS NOT NULL", con)
        con.close()
        if len(settled) > 0:
            bankroll = cfg["bankroll"] + settled["profit_loss"].sum()
    except Exception:
        pass

    print("="*70)
    print(f"  K PRO v1 -- {today}  |  Bankroll: ${bankroll:,.0f}")
    print("="*70)

    ou_sigs = signals[signals["bet_type"].isin(["ou_over", "ou_under"])].copy()
    if len(ou_sigs) > 0:
        print(f"  {'Pitcher':<20} {'Tm':<4} {'Opp':<4} {'Line':>5} {'Proj':>5}  {'Side':<6}  {'Edge':>7}  {'Stake':>6}  Sig")
        print("  " + "-"*65)
        for _, row in ou_sigs.iterrows():
            side   = "OVER"  if row["bet_type"] == "ou_over"  else "UNDER"
            edge_s = f"{row['edge']*100:+.1f}%" if row["edge"] is not None else "  --"
            stake_s = f"${row['stake']:.0f}" if row["stake"] > 0 else "  --"
            k_line  = f"{row['k_line']:.1f}" if row["k_line"] is not None else "  -"
            proj_k  = f"{row['proj_k']:.1f}" if row["proj_k"] is not None else "  -"

            print(f"  {row['pitcher'][:20]:<20} {str(row.get('team',''))[:4]:<4} "
                  f"{str(row.get('opp',''))[:4]:<4} {k_line:>5} {proj_k:>5}  "
                  f"{side:<6}  {edge_s:>7}  {stake_s:>6}  {row['signal']}")

    # Ladder
    ldr_sigs = signals[~signals["bet_type"].isin(["ou_over", "ou_under", "pass"])].copy()
    if len(ldr_sigs) > 0:
        print("")
        print("LADDER OPPORTUNITIES:")
        for pitcher, grp in ldr_sigs.groupby("pitcher"):
            team_s = str(grp["team"].iloc[0]) if "team" in grp.columns else ""
            print(f"  {pitcher} ({team_s}):")
            for _, row in grp.iterrows():
                edge_s  = f"{row['edge']*100:+.1f}%" if row["edge"] is not None else "--"
                stake_s = f"${row['stake']:.0f}" if row["stake"] > 0 else "--"
                rung_label = str(row["bet_type"]).replace("k_", "").replace("plus", "+ Ks")
                print(f"    {rung_label:<10}  odds={row['odds']:>5}  "
                      f"edge={edge_s:>7}  {stake_s:>6}  {row['signal']}")

    n_bet  = (signals["signal"] == "BET").sum()
    n_pass = (signals["signal"] == "PASS").sum()
    dist_types = signals["dist"].unique() if "dist" in signals.columns else ["?"]
    print("")
    print(f"  {n_bet} BET | {n_pass} PASS | paper={paper} | dist={','.join([str(d) for d in dist_types])}")
    print("="*70)


def quick_refresh(cfg: dict):
    """
    1. Scrape DK K props
    2. Generate signals
    3. Print dashboard
    4. Log bets
    Returns signals DataFrame.
    """
    print("=== Quick Refresh ===")
    odds_df = scrape_dk_k_props(cfg)
    signals = generate_signals(cfg, odds_df=odds_df)
    if len(signals) > 0:
        print_dashboard(signals, cfg)
        log_bets(signals, cfg)
    else:
        print("[INFO] No signals generated.")
    return signals


# ── MANUAL RUN — uncomment to execute ─────────────────────────────────────────
# signals = quick_refresh(cfg)
print("Section 12a loaded. Call quick_refresh(cfg) to run.")

## Section 12b — Daily Maintenance

Run 60–90 min before first pitch. DK removes games once started.
**Manual run only.**

In [ ]:
def daily_maintenance(cfg: dict):
    """
    Full daily pipeline:
    1. settle_bets()             — settle yesterday
    2. load_statcast_k()         — load fresh Statcast master
    3. build_pitcher_k_features() — rebuild pitcher rolling stats
    4. build_lineup_k_features()  — rebuild opponent K vulnerability
    5. build_feature_table()      — join all features
    6. quick_refresh()            — scrape odds + generate signals
    7. bet_summary()              — print tracker state
    """
    print("=" * 60)
    print(f"  K PRO v1 — DAILY MAINTENANCE  |  {date.today()}")
    print("=" * 60)
    t0 = time.time()

    print("\n[1/7] Settling yesterday's bets...")
    settle_bets(cfg)

    print("\n[2/7] Loading Statcast master...")
    load_statcast_k(cfg)

    print("\n[3/7] Building pitcher K features...")
    build_pitcher_k_features(cfg)

    print("\n[4/7] Building lineup K features...")
    build_lineup_k_features(cfg)

    print("\n[5/7] Building feature join table...")
    build_feature_table(cfg)

    print("\n[6/7] Quick refresh (odds scrape + signals)...")
    quick_refresh(cfg)

    print("\n[7/7] Bet tracker summary...")
    bet_summary(cfg)

    elapsed = time.time() - t0
    print(f"\n  Daily maintenance complete in {elapsed:.1f}s")
    print("  Timing note: Run 60–90 min before first pitch. "
          "DK removes games once started.")


# ── MANUAL RUN — uncomment to execute ─────────────────────────────────────────
# daily_maintenance(cfg)
print("Section 12b loaded. Call daily_maintenance(cfg) to run.")
print("")
print("Notebook fully loaded. System ready.")
print(f"  paper_mode = {cfg['paper_mode']}  (gate: 200 settled bets)")
print(f"  bankroll   = ${cfg['bankroll']:,.0f}")
print(f"  model_prod = {cfg['model_prod']}")